<a href="https://colab.research.google.com/github/eisbetterthanpi/vision/blob/main/chimera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title data
import torch
import torchvision
import torchvision.transforms as transforms
# transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
transform = transforms.Compose([transforms.ToTensor(),])

# CIFAR10: 60000 32x32 color images in 10 classes, with 6000 images per class
train_dataset = torchvision.datasets.CIFAR10(root='data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='data', train=False, download=True, transform=transform)
batch_size = 128 # 4
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

import numpy as np
import matplotlib.pyplot as plt
def imshow(img):
    # img = img / 2 + 0.5  # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# dataiter = iter(train_loader) # get some random training images
# images, labels = next(dataiter)
# print(images.shape) # [batch, 3, 32, 32]
# imshow(torchvision.utils.make_grid(images))


In [ ]:

for each vert, 4 diag directions, 2 for hori/vert edge
for mask, let aij=1, bij=0, cij=1? no compression


In [ ]:
# @title chimera utils
# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/src/models/sequence/modules/chimera.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class DAGInverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, A, k): # A:bnll
        I = torch.eye(A.size(-1), dtype=A.dtype, device=A.device)[None,None,...]
        IA = I + A
        for _ in range(2, math.ceil(math.log2(k))+1): # Compute (I+A)(I+A^2)(I+A^4)...(I+A^j) # TODO: Do this in log(j) steps, but for j = 32, O(j) is fine
            A = A @ A
            IA = torch.matmul(IA, I + A)
        ctx.save_for_backward(IA)
        return IA # = I+A+A^2+A^3+...+A^(2j-1)

    @staticmethod
    def backward(ctx, grad_output):
        IA = ctx.saved_tensors[0].transpose(-2,-1) # Retrieve the saved tensors from the forward pass
        grad_A = IA @ (grad_output @ IA) # Compute the gradient with respect to A
        return grad_A, None

def make_num_edge(h,w):
    num_edge = 2*torch.ones(2,h,w, device=device) # mid got 4, walls got 3, corners got 2
    num_edge[0,:,0] = num_edge[1,:,-1] = 1 # l,r
    num_edge = num_edge.repeat(2,1,1) # 4hw # l,r,l,r
    num_edge[:2,0] = num_edge[2:,-1] = 1 # 4hw # lt,rt,lb,rb
    return num_edge.flatten(1) # 4t

def make_adj_mat(weight, h,w): # bnt
    *b,t = weight.shape
    l,r,u,d = [torch.zeros(*b,t,t, device=device) for _ in range(4)]
    index = torch.arange(t, device=device).reshape(h,w)
    left, right = index[:,:-1].flatten(), index[:,1:].flatten()
    top, bottom = index[:-1].flatten(), index[1:].flatten()
    # adj_mat[:,:,start,end]
    # l[...,right,left] = weight[...,right] # left
    # r[...,left,right] = weight[...,left] # right
    # u[...,bottom,top] = weight[...,bottom] # top
    # d[...,top,bottom] = weight[...,top] # bottom
    l[...,right,left] = weight[...,left] # left
    r[...,left,right] = weight[...,right] # right
    u[...,bottom,top] = weight[...,top] # top
    d[...,top,bottom] = weight[...,bottom] # bottom
    adj_mat = torch.stack([l+u, r+u, l+d, r+d], dim=0) # 4bntt
    # adj_mat = adj_mat / torch.sqrt(num_edge.unsqueeze(-1)) # bntt/t1 # is_normalize T # each dir contribute 1/sqrt(num_dirs) magnitude
    return adj_mat # 4bntt


h,w = 2,2
# h,w = 3,3
t=h*w

# b,n = 1,1
b,n = 2,3
weight = torch.randn(b,n,t, device=device)
adj_mat = make_adj_mat(weight, h,w)
# print(adj_mat)
# num_edge = make_num_edge(h,w)
# print(num_edge)

# A:top_left
# 0000
# 1000
# 1000
# 0110
# 0<1
# ^ ^
# 2<3


In [ ]:
# @title chimera one works
# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/src/models/sequence/modules/chimerablock.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# @torch.compile()
class Chimera(nn.Module):
    def __init__(self, d_model, expand=2, n_heads=8, n_groups=8, d_state=8, d_conv=2, height=8, width=8):
        super().__init__()
        n_groups = min(n_heads, n_groups)
        assert n_heads % n_groups == 0, "nheads must be divisible by ngroups"
        self.d_model, self.n_groups, self.d_state, = d_model, n_groups, d_state
        self.d_inner = expand * d_model
        self.n_heads, self.d_head = n_heads, self.d_inner//n_heads
        self.height, self.width = height, width

        self.in_proj = nn.Linear(d_model, 2* self.d_inner + 2* self.n_groups*self.d_state + self.n_heads, bias=False)
        conv_dim = self.d_inner + 2* self.n_groups*self.d_state
        self.conv2d = nn.Conv2d(conv_dim, conv_dim, kernel_size=d_conv*2-1, groups=conv_dim, padding=d_conv-1, bias=True)
        self.act = nn.SiLU()

        # # Mask for directed convolution
        # conv_mask = torch.zeros(d_conv*2-1, d_conv*2-1)
        # conv_mask[0:d_conv, 0:d_conv] = torch.flip(torch.tril(torch.ones(d_conv, d_conv)), dims=[1])
        # conv_mask[0:d_conv, d_conv-1:] = torch.tril(torch.ones(d_conv, d_conv))
        # conv_mask[d_conv-1:, 0:d_conv] = torch.triu(torch.ones(d_conv, d_conv))
        # conv_mask[d_conv-1:, d_conv-1:] = torch.flip(torch.triu(torch.ones(d_conv, d_conv)), dims=[1])
        # nn.init.uniform_(self.conv1d.weight, -conv_init, conv_init)
        # self.conv2d.weight.data *= conv_mask

        dt_min, dt_max = .001, .1
        dt = torch.exp(torch.rand(self.n_heads) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt))) # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        self.dt_bias._no_reinit, self.dt_bias._no_weight_decay = True, True

        self.D = nn.Parameter(torch.ones(self.n_heads, device=device))
        self.D._no_reinit, self.D._no_weight_decay = True, True
        self.norm = nn.RMSNorm(self.d_inner, eps=1e-5)
        # self.dropout = DropoutNd(0, transposed=False) # tie_dropout # from src.models.nn import DropoutNd
        self.dropout = nn.Dropout(0)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        self.num_edge = make_num_edge(height, width)[:,None,None,:] # 4t->411t1

    def forward(self, u): # btd
        t = u.size(-2)
        z, xBC, dt = self.in_proj(u).split([self.d_inner, self.d_inner + 2*self.n_groups*self.d_state, self.n_heads], dim=-1)
        xBC = xBC.transpose(-2,-1).unflatten(-1, (self.height, self.width)) # b(hw)d->bdhw
        xBC = self.act(self.conv2d(xBC))
        xBC = xBC.flatten(2).transpose(1,2) # bdhw->btd
        x, B, C = xBC.split([self.d_inner, self.n_groups*self.d_state, self.n_groups*self.d_state], dim=-1)
        x, B, C = x.unflatten(-1, (self.n_heads, self.d_head)), B.unflatten(-1, (self.n_groups, self.d_state)), C.unflatten(-1, (self.n_groups, self.d_state)) # x:[b,t,h,d], B/C:[b,t,g,s]
        h_g = self.n_heads//self.n_groups
        if h_g>1: B, C = B.repeat_interleave(h_g, dim=-2), C.repeat_interleave(h_g, dim=-2) # btgs->bths
        x, B, C, dt = [a.transpose(1,2) for a in (x, B, C, dt)] # X:bhtd, B/C:bhts
        dt = F.softplus(dt + self.dt_bias.unsqueeze(-1)) # bnt+n1->bnt # aka ln(Aij)?
        dA = torch.exp(-dt) # bnt # ~Abar in mamba

        A = make_adj_mat(dA, self.height, self.width) / torch.sqrt(self.num_edge.unsqueeze(-1)) # 4bntt/411t1 # is_normalize T # each dir contribute 1/sqrt(num_dirs) magnitude
        # print('A, num_incident_edges', A.shape, num_incident_edges.shape)
        L = DAGInverse.apply(A, self.height+self.width) # use_fast_inverse=T
        # L = torch.inverse(torch.eye(self.height*self.width, device=device) - A) # use_fast_inverse=F

        # print('dt, self.num_edge', dt.shape, self.num_edge.shape)
        # norm_dt = (1-torch.exp(2*dt)).expand(4,-1,-1,-1)/torch.sqrt(self.num_edge) # 4bnt/411t=4bnt
        norm_dt = dt.expand(4,-1,-1,-1)/torch.sqrt(self.num_edge) # 4bnt/411t=4bnt
        # norm_dt = dt # bnt
        # # print('B, norm_dt', B.shape, norm_dt.shape)
# h = Ah + Bx : A*h + x@B = 1/ds*ds + d1@1s = ds
# y = Ch + Dx : h@C + D*x = ds@s1 + 1/d1*d1 = d1

        # dB = torch.einsum('bnls,bnl->bnls', B, norm_dt) # [b,n,t,qk_dim]*bnt
        # # dB = torch.einsum('bnls,fbnl->bnls', B, norm_dt) # [b,n,t,qk_dim]*4bnt
        # y = torch.einsum('bnls,bnlt,bnts,bntd->bnld', C, L, dB, x)
        # L = torch.cat([A, L])
        y = torch.einsum('bnls,fbnlt,bnls,fbnl,bntd->bnld', C, L, B, norm_dt, x)

        # dBx = torch.einsum('bnls,fbnl,bnld->fbnlds', B, norm_dt, x)
        # h_ssm = torch.einsum('fbnlt,bnls,fbnl,bntd->bnlds', L, B, norm_dt, x)
        y = y/2 # y*4**-.5, 4dirs
        # print('y',y[0,0,-1,:5])
        # print('x',x[0,0,-1,:5])
        print('x y',(x**2).mean().item(), (y**2).mean().item())
        y = y + x*self.D[:,None,None] # bntd # std_dev*y+hid, std_dev=1 This will be per method parameter
        y = y.transpose(1,2).flatten(2) # [b,t,d]/[1,b*t,d]
        y = self.norm(y * F.silu(z)) # self.norm(y) * F.silu(z)
        y = self.out_proj(self.dropout(y))
        return y[:, :t, :], None

# <^ /sqrt2
# 11
# 12
# (<^+^>+<v+v>) /sqrt2 ; /sqrt4
# =2(<+^+>+v)/sqrt2
# norm A so that along the path is all 1. norm dBx

# let weight be end of path, not start

# ^
# 0000
# 0000
# 1000
# 0100
# v
# 0010
# 0001
# 0000
# 0000
# >
# 0100
# 0000
# 0001
# 0000
# <
# 0000
# 1000
# 0000
# 0010
d=64
b=3
h,w = 14,14
model = Chimera(d, height=h, width=w).to(device)
x = torch.randn(b, h*w, d, device=device)
y, _ = model(x)
# y = model(x)
print(y.shape)



In [ ]:
# @title Chimera Block ViT
import torch
import torch.nn as nn
device = 'cuda' if torch.cuda.is_available() else 'cpu'

import inspect
class Seq(nn.Sequential):
    def __init__(self, *args):
        super().__init__(*args)
    def forward(self, x, hid=None):
        hidden = []
        for i, layer in enumerate(self):
            x, h = layer(x, hid[i] if hid!=None else None)
            hidden.append(h)
        return x, hidden

# class Seq(nn.Sequential):
#     def __init__(self, *args):
#         super().__init__(*args)
#         self._kwargs = [[name for name, p in inspect.signature(layer.forward).parameters.items()] for layer in self]

#     # def forward(self, x, *args, **kwargs):
#     #     for layer, _kwargs in zip(self, self._kwargs):
#     #         x = layer(x, *args, **{k: v for k, v in kwargs.items() if k in _kwargs})
#     #     return x
# #     def forward(self, x, hid=None, *args, **kwargs):
# #         hidden = []
# #         for i, layer in enumerate(self):
# #             x, h = layer(x, hid[i] if hid!=None else None)
# #             hidden.append(h)
# #         return x, hidden
#     def forward(self, x):
#         for layer, _kwargs in zip(self, self._kwargs):
#             x = layer(x, *args, **{k: v for k, v in kwargs.items() if k in _kwargs})
#         return x

class Block(nn.Module):
    def __init__(self, d_model, n_heads=None, drop=0):
    # def __init__(self, d_model, drop=0, *args, **kwargs):
        super().__init__()
        self.d_model = d_model
        self.norm1 = nn.RMSNorm(d_model) # LayerNorm RMSNorm
        # self.norm2 = nn.RMSNorm(d_model)
        # self.drop = nn.Dropout(drop)
        # self.rnn = Hydra(d_model, n_heads=n_heads)
        # self.rnn = Chimera(d_model, args, kwargs)
        self.rnn = Chimera(d_model)

    def forward(self, x, h=None): # [b,t,d], (,)
        # print('blk fwd', x.shape, h.shape if h is not None else None)
        # x_, h = self.rnn(self.norm1(x), h) #
        x_, h = self.rnn(self.norm1(x)) #
        x = x + x_
        return x, h

class SimpleViT(nn.Module):
    def __init__(self, in_dim, d_model, out_dim=None, n_heads=4, nlyrs=1):
        super().__init__()
        # self.embed = nn.Linear(in_dim, d_model)
        self.embed = nn.Sequential( # in, out, kernel, stride, pad
            nn.Conv2d(in_dim, d_model, 7, 2, 7//2, bias=False), nn.BatchNorm2d(d_model), nn.ReLU(),
            nn.Conv2d(d_model, d_model, 3, 2, 3//2, bias=False)
            )
        # self.blk = Seq(*[Block(d_model, n_heads) for _ in range(nlyrs)])
        # self.blk = Seq(*[Block(d_model, height=8, width=8) for _ in range(nlyrs)])
        self.blk = Seq(*[Block(d_model) for _ in range(nlyrs)])
        self.attn_pool = nn.Linear(d_model, 1)
        self.out = nn.Linear(d_model, out_dim or d_model, bias=False)

    def forward(self, x, mask=None): # [b,c,h,w]
        x = self.embed(x)
        x = x.flatten(2).transpose(1,2) # [b,h*w,c]
        # print('vit fwd', x.shape)
        x,_ = self.blk(x)
        attn = self.attn_pool(x) # [b,h*w,1] # seq_pool
        x = (attn.softmax(dim=1).transpose(-2,-1) @ x).squeeze(1) # [b,1,h*w] @ [b,h*w,d] -> [b,d]
        return self.out(x)

# # b,t,d = 2,256,16
# b,t,d = 2,7,16
# x = torch.rand(b,t,d, device=device)
# model = Block(d_model=d, n_heads=4).to(device)
# # model = Seq(*[Block(d_model=d, n_heads=4) for _ in range(2)])
# out, h = model(x)
# print(out.shape)

dim = 64#64
in_dim=3
out_dim = 10
model = SimpleViT(in_dim, 64, out_dim, nlyrs=1, n_heads=8).to(device) # 64129
print(sum(p.numel() for p in model.parameters() if p.requires_grad)) # 59850
optim = torch.optim.AdamW(model.parameters(), lr=1e-3)

# print(images.shape) # [batch, 3, 32, 32]
x = torch.rand(24, 3, 32,32, device=device)
# x = torch.rand(64, 3, 28,28, device=device)
logits = model(x)
print(logits.shape)

# hydra lin 47257p Accuracy: 54.1%, Avg loss: 1.276877 time: 1.9073486328125e-06 118.25601513385773
# hydra conv 93401p Accuracy: 69.5%, Avg loss: 0.896258 time: 1.6689300537109375e-06 17.178589010238646
# trans ggrope lin 50945p Accuracy: 50.6%, Avg loss: 1.364122 time: 1.6689300537109375e-06 26.679788732528685
# trans ggrope conv 97089p Accuracy: 67.3%, Avg loss: 0.948646 time: 1.6689300537109375e-06 17.618668341636656

# Hydra one BCdt lin 37529p
# Hydra one BCdt expand3 lin 50905p Accuracy: 54.8%, Avg loss: 1.263319 time: 1.6689300537109375e-06 123.11688303947449

# hydra conv 83673
# hydra conv mamba2me conv(u) <96033p Accuracy: 68.9%, Avg loss: 0.928908 2.86102294921875e-06 20.047950315475465

# 92549p Accuracy: 66.1%, Avg loss: 0.986353 time: 1.430511474609375e-06 17.032868480682374
# 82885p Accuracy: 66.7%, Avg loss: 0.987715 time: 1.1920928955078125e-06 16.369480323791503
# chimera one 83153p Accuracy: 68.1%, Avg loss: 0.923436 time: 1.9073486328125e-06 20.11947512626648
# chimera make_adj_mat +=dt,+=L Accuracy: 70.4%, Avg loss: 0.863932 time: 2.384185791015625e-06 16.036279177665712
#  Accuracy: 69.0%, Avg loss: 0.916525 time: 1.9073486328125e-06 17.865441966056824



In [ ]:
# @title wandb
!pip install -q wandb
import wandb # https://docs.wandb.ai/quickstart
wandb.login(key='487a2109e55dce4e13fc70681781de9f50f27be7')
try: run.finish()
except NameError: pass
run = wandb.init(project="vit", config={"model": "chimera",})

In [ ]:
# @title train test function
import torch.nn.functional as F
# from torch.utils.tensorboard import SummaryWriter
# writer = SummaryWriter()
# global_step=0

def train(dataloader, model, optim):
    size = len(dataloader.dataset)
    model.train()
    for i, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = F.cross_entropy(pred, y)
        optim.zero_grad()
        loss.backward()
        optim.step()
        if i % (size//(10* len(x))) == 0:
            loss, current = loss.item(), i * len(x)
            # loss_list.append(loss)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        try: wandb.log({"loss": loss.item()})
        except: pass
        # global global_step
        # writer.add_scalars('train', {"loss": loss}, global_step=global_step, walltime=None) # https://docs.pytorch.org/docs/stable/tensorboard.html#torch.utils.tensorboard.writer.SummaryWriter.add_scalar
        # global_step+=1

def test(dataloader, model):
    model.eval()
    test_loss, correct = 0, 0
    for X, y in dataloader:
        x, y = X.to(device), y.to(device)
        with torch.no_grad(): pred = model(x)
        test_loss += F.cross_entropy(pred, y).item()
        correct += (pred.argmax(1) == y).sum().item()
    test_loss /= len(dataloader)
    correct /= len(dataloader.dataset)
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    try: wandb.log({"test_loss": test_loss, "accuracy": 100*correct})
    except: pass
    # global global_step
    # writer.add_scalars('test', {"acc": 100*correct, "loss": test_loss}, global_step=global_step, walltime=None) # https://docs.pytorch.org/docs/stable/tensorboard.html#torch.utils.tensorboard.writer.SummaryWriter.add_scalar
    return correct

import time
start = time.time()
for t in range(10):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_loader, model, optim)
    test(test_loader, model)
    end = time.time()
    print('time:',time.time() - end, (time.time()-start)/(t+1))

end = time.time()
print("time: ",end - start)
# writer.close()


In [ ]:
# @title save/load
from google.colab import drive
drive.mount('/content/drive')
folder='/content/drive/MyDrive/jepa/'

# modelsd, optimsd = torch.load(folder+'hydra.pkl', map_location=device).values()
# model.load_state_dict(modelsd, strict=False)
# optim.load_state_dict(optimsd)

In [ ]:
# checkpoint = {'model': model.state_dict(), 'optimizer': optim.state_dict()}
# torch.save(checkpoint, folder+'hydra.pkl')
# # torch.save(checkpoint, 'hydra.pkl')

## old

In [ ]:
# @title grid_adj_nd
import torch

def grid_adj_nd(shape):
    # shape = torch.tensor(shape, device=device)
    shape = torch.tensor(shape)
    n = len(shape)
    t = int(shape.prod())
    strides = torch.cat([torch.ones(1), torch.cumprod(shape[1:].flip(0), dim=0).flip(0)])
    rows, cols = [], []
    verts = torch.arange(t, device=device)
    for dim in range(n):
        stride = strides[dim] # [1,3]
        size = shape[dim]
        coord = (verts // stride) % size
        mask_p = coord < size - 1 # +1 neighbour
        rows.append(verts[mask_p])
        cols.append(verts[mask_p] + stride)
        mask_m = coord > 0 # -1 neighbour
        rows.append(verts[mask_m])
        cols.append(verts[mask_m] - stride)
    indices = torch.stack([torch.cat(rows), torch.cat(cols)])
    # values = torch.ones(indices.shape[1], device=device, dtype=dtype)
    values = torch.ones(indices.shape[1], device=device)
    return torch.sparse_coo_tensor(indices, values, (t,t))

h,w = 3,3
A = grid_adj_nd([h,w]) # [h*w, h*w]
# m = grid_adj_nd([5,5,3])
# print(m)
# print(m.to_dense())
# A = m.to_dense()
# A = A.flatten()
# A = A.to(bool)
# print(A.shape)
L = torch.inverse(torch.eye(h*w, device=device) - A) # use_fast_inverse=F
# print(L.shape)
print(L)
d=4
dBx = torch.randn(h*w,d, device=device)
# y = torch.einsum('bnls,fbnlt,bnls,fbnl,bntd->bnld', L, dBx)
y = torch.einsum('tl,td->ld', L, dBx)
print(y)


In [ ]:
# @title chimera one works
# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/src/models/sequence/modules/chimerablock.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# @torch.compile()
class Chimera(nn.Module):
    def __init__(self, d_model, expand=2, n_heads=8, n_groups=8, d_state=8, d_conv=2, height=8, width=8):
        super().__init__()
        n_groups = min(n_heads, n_groups)
        assert n_heads % n_groups == 0, "nheads must be divisible by ngroups"
        self.d_model, self.n_groups, self.d_state, = d_model, n_groups, d_state
        self.d_inner = expand * d_model
        self.n_heads, self.d_head = n_heads, self.d_inner//n_heads
        self.height, self.width = height, width

        self.in_proj = nn.Linear(d_model, 2* self.d_inner + 2* self.n_groups*self.d_state + self.n_heads, bias=False)
        conv_dim = self.d_inner + 2* self.n_groups*self.d_state
        self.conv2d = nn.Conv2d(conv_dim, conv_dim, kernel_size=d_conv*2-1, groups=conv_dim, padding=d_conv-1, bias=True)
        self.act = nn.SiLU()

        # # Mask for directed convolution
        # conv_mask = torch.zeros(d_conv*2-1, d_conv*2-1)
        # conv_mask[0:d_conv, 0:d_conv] = torch.flip(torch.tril(torch.ones(d_conv, d_conv)), dims=[1])
        # conv_mask[0:d_conv, d_conv-1:] = torch.tril(torch.ones(d_conv, d_conv))
        # conv_mask[d_conv-1:, 0:d_conv] = torch.triu(torch.ones(d_conv, d_conv))
        # conv_mask[d_conv-1:, d_conv-1:] = torch.flip(torch.triu(torch.ones(d_conv, d_conv)), dims=[1])
        # nn.init.uniform_(self.conv1d.weight, -conv_init, conv_init)
        # self.conv2d.weight.data *= conv_mask

        dt_min, dt_max = .001, .1
        dt = torch.exp(torch.rand(self.n_heads) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt))) # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        self.dt_bias._no_reinit, self.dt_bias._no_weight_decay = True, True

        self.D = nn.Parameter(torch.ones(self.n_heads, device=device))
        self.D._no_reinit, self.D._no_weight_decay = True, True
        self.norm = nn.RMSNorm(self.d_inner, eps=1e-5)
        # self.dropout = DropoutNd(0, transposed=False) # tie_dropout # from src.models.nn import DropoutNd
        self.dropout = nn.Dropout(0)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        self.num_edge = make_num_edge(height, width)[:,None,None,:] # 4t->411t1

    def forward(self, u): # btd
        t = u.size(-2)
        z, xBC, dt = self.in_proj(u).split([self.d_inner, self.d_inner + 2*self.n_groups*self.d_state, self.n_heads], dim=-1)
        xBC = xBC.transpose(-2,-1).unflatten(-1, (self.height, self.width)) # b(hw)d->bdhw
        xBC = self.act(self.conv2d(xBC))
        xBC = xBC.flatten(2).transpose(1,2) # bdhw->btd
        x, B, C = xBC.split([self.d_inner, self.n_groups*self.d_state, self.n_groups*self.d_state], dim=-1)
        x, B, C = x.unflatten(-1, (self.n_heads, self.d_head)), B.unflatten(-1, (self.n_groups, self.d_state)), C.unflatten(-1, (self.n_groups, self.d_state)) # x:[b,t,h,d], B/C:[b,t,g,s]
        h_g = self.n_heads//self.n_groups
        if h_g>1: B, C = B.repeat_interleave(h_g, dim=-2), C.repeat_interleave(h_g, dim=-2) # btgs->bths
        x, B, C, dt = [a.transpose(1,2) for a in (x, B, C, dt)] # X:bhtd, B/C:bhts
        dt = F.softplus(dt + self.dt_bias.unsqueeze(-1)) # bnt+n1->bnt # aka ln(Aij)?
        dA = torch.exp(-dt) # bnt # ~Abar in mamba

        A = make_adj_mat(dA, self.height, self.width) / torch.sqrt(self.num_edge.unsqueeze(-1)) # 4bntt/411t1 # is_normalize T # each dir contribute 1/sqrt(num_dirs) magnitude
        # print('A, num_incident_edges', A.shape, num_incident_edges.shape)
        L = DAGInverse.apply(A, self.height+self.width) # use_fast_inverse=T
        # L = torch.inverse(torch.eye(self.height*self.width, device=device) - A) # use_fast_inverse=F

        # print('dt, self.num_edge', dt.shape, self.num_edge.shape)
        # norm_dt = (1-torch.exp(2*dt)).expand(4,-1,-1,-1)/torch.sqrt(self.num_edge) # 4bnt/411t=4bnt
        norm_dt = dt.expand(4,-1,-1,-1)/torch.sqrt(self.num_edge) # 4bnt/411t=4bnt
        # norm_dt = dt # bnt
        # # print('B, norm_dt', B.shape, norm_dt.shape)
# h = Ah + Bx : A*h + x@B = 1/ds*ds + d1@1s = ds
# y = Ch + Dx : h@C + D*x = ds@s1 + 1/d1*d1 = d1

        y = torch.einsum('bnls,fbnlt,bnls,fbnl,bntd->bnld', C, L, B, norm_dt, x)

        y = y/2 # y*4**-.5, 4dirs
        # print('y',y[0,0,-1,:5])
        # print('x',x[0,0,-1,:5])
        print('x y',(x**2).mean().item(), (y**2).mean().item())
        y = y + x*self.D[:,None,None] # bntd # std_dev*y+hid, std_dev=1 This will be per method parameter
        y = y.transpose(1,2).flatten(2) # [b,t,d]/[1,b*t,d]
        y = self.norm(y * F.silu(z)) # self.norm(y) * F.silu(z)
        y = self.out_proj(self.dropout(y))
        return y[:, :t, :], None

# <^ /sqrt2
# 11
# 12
# (<^+^>+<v+v>) /sqrt2 ; /sqrt4
# =2(<+^+>+v)/sqrt2
# norm A so that along the path is all 1. norm dBx

let weight be end of path, not start

# ^
# 0000
# 0000
# 1000
# 0100
# v
# 0010
# 0001
# 0000
# 0000
# >
# 0100
# 0000
# 0001
# 0000
# <
# 0000
# 1000
# 0000
# 0010
d=64
b=3
h,w = 14,14
model = Chimera(d, height=h, width=w).to(device)
x = torch.randn(b, h*w, d, device=device)
y, _ = model(x)
# y = model(x)
print(y.shape)



In [ ]:
# @title grid_adj mat store
import torch

def grid_adj_nd(shape, device=None, dtype=torch.float32):
    # shape = torch.tensor(shape, device=device)
    shape = torch.tensor(shape)
    n = len(shape)
    V = int(shape.prod())
    strides = torch.cat([torch.ones(1), torch.cumprod(shape[1:].flip(0), dim=0).flip(0)])
    rows, cols = [], []
    # print('strides',strides)
    # flattened vertex ids
    verts = torch.arange(V, device=device)
    for dim in range(n):
        stride = strides[dim] # [1,3]
        size = shape[dim]

        # mask to avoid crossing boundaries
        coord = (verts // stride) % size
        # print('coord',coord)
# [0., 1., 2., 0., 1., 2., 0., 1., 2.])
# [0., 0., 0., 1., 1., 1., 2., 2., 2.]
        # +1 neighbour
        mask_p = coord < size - 1 # all verts that are not at extreme right/bottom/back (depending on dim)
        # print(mask_p)
        # print(verts[mask_p])
        # print(verts[mask_p] + stride)
        rows.append(verts[mask_p])
        cols.append(verts[mask_p] + stride) # right/bottom/back

        # -1 neighbour
        mask_m = coord > 0 # all verts that are not at extreme left/top/front
        print(mask_m)
        print(verts[mask_m])
        print(verts[mask_m] - stride)
        rows.append(verts[mask_m])
        cols.append(verts[mask_m] - stride)
        # its basically: for last dim, add all left and right (with its corresponding row), for 2nd last dim, add all up and down, ...
    indices = torch.stack([torch.cat(rows), torch.cat(cols)])
    values = torch.ones(indices.shape[1], device=device, dtype=dtype)
    return torch.sparse_coo_tensor(indices, values, (V, V))

m = grid_adj_nd([3,3])
# m = grid_adj_nd([5,5,3])
# print(m)
# print(m.to_dense())

# 2d
import torch

def dense_adj(shape):
    shape = list(shape)
    V = int(torch.tensor(shape).prod())
    A = torch.zeros(V, V)
    strides = torch.cumprod(torch.tensor([1] + shape[:0:-1]), 0).flip(0)
    # verts = torch.arange(V, device=device)
    verts = torch.arange(V)
    for dim, stride in enumerate(strides):
        size = shape[dim]
        coord = (verts // stride) % size
        mask_p = coord < size - 1
        mask_m = coord > 0
        A[verts[mask_p], verts[mask_p] + stride] = 1
        A[verts[mask_m], verts[mask_m] - stride] = 1
    return A

print(dense_adj([3,3]))



In [ ]:
# @title chimera utils one & sparse
# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/src/models/sequence/modules/chimera.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class DAGInverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, A, k): # A:bnll
        I = torch.eye(A.size(-1), dtype=A.dtype, device=A.device)[None,None,...]
        IA = I + A
        for _ in range(2, math.ceil(math.log2(k))+1): # Compute (I+A)(I+A^2)(I+A^4)...(I+A^j) # TODO: Do this in log(j) steps, but for j = 32, O(j) is fine
            A = A @ A
            IA = torch.matmul(IA, I + A)
        ctx.save_for_backward(IA)
        return IA # = I+A+A^2+A^3+...+A^(2j-1)

    @staticmethod
    def backward(ctx, grad_output):
        IA = ctx.saved_tensors[0].transpose(-2,-1) # Retrieve the saved tensors from the forward pass
        grad_A = IA @ (grad_output @ IA) # Compute the gradient with respect to A
        return grad_A, None

def make_adj_mat(weight, h,w): # bnt
    b,n,t = weight.shape
    adj_mat = torch.zeros(b,n,t,t, device=device)
    index = torch.arange(t, device=device).reshape(h,w)
    left, right = index[:,:-1].flatten(), index[:,1:].flatten()
    top, bottom = index[:-1].flatten(), index[1:].flatten()
    # adj_mat[:,:,start,end]
    adj_mat[:,:,right,left] += weight[:,:,right] # left
    adj_mat[:,:,left,right] += weight[:,:,left] # right
    adj_mat[:,:,bottom,top] += weight[:,:,bottom] # top
    adj_mat[:,:,top,bottom] += weight[:,:,top] # bottom
    # adj_mat = adj_mat / torch.sqrt(num_edge.unsqueeze(-1)) # bntt/t1 # is_normalize T # each dir contribute 1/sqrt(num_dirs) magnitude
    return adj_mat # bntt

def make_num_edge(h,w):
    num_edge = 4*torch.ones(h, w, device=device) # mid got 4, walls got 3, corners got 2
    num_edge[0]=num_edge[-1]=num_edge[:,0]=num_edge[:,-1]=3
    num_edge[0,0]=num_edge[-1,-1]=num_edge[0,-1]=num_edge[-1,0]=2
    return num_edge.flatten() # t

def grid_adj_nd(shape, weight):
    # shape = torch.tensor(shape, device=device)
    shape = torch.tensor(shape)
    n = len(shape)
    t = int(shape.prod())
    strides = torch.cat([torch.ones(1), torch.cumprod(shape[1:].flip(0), dim=0).flip(0)])
    rows, cols = [], []
    vals = []
    verts = torch.arange(t, device=device)
    for dim in range(n):
        stride = strides[dim] # [1,3]
        size = shape[dim]
        coord = (verts // stride) % size
        mask_p = coord < size - 1 # +1 neighbour
        rows.append(verts[mask_p])
        cols.append(verts[mask_p] + stride)
        vals.append(weight[:,:,mask_p])
        mask_m = coord > 0 # -1 neighbour
        rows.append(verts[mask_m])
        cols.append(verts[mask_m] - stride)
        vals.append(weight[:,:,mask_m])
        # print(weight[:,:,mask_m].shape) # [bnt?]
    rows, cols = torch.cat(rows), torch.cat(cols) # [num edges]
    values = torch.cat(vals, dim=-1) # [b,n,num edges]
    # print(rows.shape, values.shape)
    b,n,t = weight.shape
    offset = (torch.arange(b*n, device=device) * t).reshape(b,n,1)
    rows, cols = rows[None,None,:] + offset, cols[None,None,:] + offset # 11e+bn1=bne
    indices = torch.stack([rows.flatten(), cols.flatten()]) # [2,b*n*e]
    values = values.flatten()
    return torch.sparse_coo_tensor(indices, values, (b*n*t,b*n*t))


# # h,w = 2,2
# h,w = 3,3
# t=h*w

# b,n = 1,1
# b,n = 2,3
# adj_mat = torch.zeros(b,n,t,t, device=device)
# weight = torch.randn(b,n,t, device=device)
# adj_mat = make_adj_mat(h,w, weight)
# # print(adj_mat)

# m = grid_adj_nd([h,w],weight)
# # m = grid_adj_nd([5,5,3])
# # print(m)
# # print(m.to_dense())
# # print(abs(adj_mat-m.to_dense()).sum())

# A:top_left
# 0000
# 1000
# 1000
# 0110
# 0<1
# ^ ^
# 2<3



In [ ]:
# @title chimera utils base
# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/src/models/sequence/modules/chimera.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class DAGInverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, A, k): # A:bnll
        I = torch.eye(A.size(-1), dtype=A.dtype, device=A.device)[None,None,...]
        IA = I + A
        for _ in range(2, math.ceil(math.log2(k))+1): # Compute (I+A)(I+A^2)(I+A^4)...(I+A^j) # TODO: Do this in log(j) steps, but for j = 32, O(j) is fine
            A = A @ A
            IA = torch.matmul(IA, I + A)
        ctx.save_for_backward(IA)
        return IA # = I+A+A^2+A^3+...+A^(2j-1)

    @staticmethod
    def backward(ctx, grad_output):
        IA = ctx.saved_tensors[0].transpose(-2,-1) # Retrieve the saved tensors from the forward pass
        grad_A = IA @ (grad_output @ IA) # Compute the gradient with respect to A
        return grad_A, None

def get_unused_dt_mask(height, width):
    # Must be used in the order [0]top_left, [1]top_right, [2]bottom_left, [3]bottom_right,
    # where each is followed by [0]left-right, [1]top-bottom
    # Visualize a Rectangle with 4 corners
    # 0 --(a)--- 1
    # |          |
    #(d)        (b)
    # |          |
    # 2 --(c)--- 3
    dt_mask = torch.ones(4, 2, height, width, device=device)
    # ones(h,w) except line of zeros at luruldrd
    # Corners and Edges:
    # For (a) remove [0][1], and [1][1]
    dt_mask[0,1,0,:] = 0
    dt_mask[1,1,0,:] = 0
    # For (b) remove [1][0], and [3][0]
    dt_mask[1,0,:,-1] = 0
    dt_mask[3,0,:,-1] = 0
    # For (c) remove [2][1], and [3][1]
    dt_mask[2,1,-1,:] = 0
    dt_mask[3,1,-1,:] = 0
    # For (d) remove [0][0], and [2][0]
    dt_mask[0,0,:,0] = 0
    dt_mask[2,0,:,0] = 0
    return dt_mask # 42hw



def make_num_edge(h,w, tol=1e-6):
    dirs = ['top_left', 'top_right', 'bottom_left', 'bottom_right']
    num_edge = 2*torch.ones(len(dirs), h, w, device=device)
    for i, dir in enumerate(dirs):
        if 'left' in dir: num_edge[i,:,0] = num_edge[i,:,0] - 1
        elif 'right' in dir: num_edge[i,:,-1] = num_edge[i,:,-1] - 1
        if 'top' in dir: num_edge[i,0] = num_edge[i,0] - 1
        elif 'bottom' in dir: num_edge[i,-1] = num_edge[i,-1] - 1
    num_edge[num_edge < tol] = 1 # Divide each row by the number of non-zero values
    return num_edge.flatten(1) # 4t

def make_adj_mat(weight, dir, tol=1e-6): # bnt
    b,n,t = weight.shape
    adj_mat = torch.zeros(b,n,t,t, device=device)
    index = torch.arange(t, device=device).reshape(h,w)
    # else: print('get_normalized_dag_mixer err')
    # adj_mat[:,:,start,end]
    left, right = index[:,:-1].flatten(), index[:,1:].flatten()
    if 'left' in dir: adj_mat[:,:,right,left] = weight[:,:,right]
    elif 'right' in dir: adj_mat[:,:,left,right] = weight[:,:,left]
    top, bottom = index[:-1].flatten(), index[1:].flatten()
    if 'top' in dir: adj_mat[:,:,bottom,top] = weight[:,:,bottom]
    elif 'bottom' in dir: adj_mat[:,:,top,bottom] = weight[:,:,top]
    # adj_mat = adj_mat / torch.sqrt(num_edge.unsqueeze(-1)) # bntt/t1 # is_normalize T # each dir contribute 1/sqrt(num_dirs) magnitude
    return adj_mat # bntt

# # adj_inds = torch.zeros(4,t,t, dtype=torch.long, device=device)
# adj_inds = -torch.ones(4,t,t, dtype=torch.long, device=device)
# # top_left
# adj_inds[0,right,left] = right
# adj_inds[0,bottom,top] = bottom
# # top_right
# adj_inds[1,left,right] = left
# adj_inds[1,bottom,top] = bottom
# # bottom_left
# adj_inds[2,right,left] = right
# adj_inds[2,top,bottom] = top
# # bottom_right
# adj_inds[3,left,right] = left
# adj_inds[3,top,bottom] = top
# for x in adj_inds:
#     print(x)


# weight bnt/bnhw
# adj inds 4tt # 4dirs, 2hori/vert no overlap, can be combined
# adj_mat 4bntt
# 4bntt[4tt] := bnt

# maybe no need inds, use slicing instead. nvm, even after slice need to convert to adj mat
# left = torch.cat([weight[:,:,:,1:], torch.zeros(b,n,h,1)], dim=-1) # bnhw
# right = torch.cat([torch.zeros(b,n,h,1), weight[:,:,:,:-1]], dim=-1)
# top = torch.cat([weight[:,:,1:], torch.zeros(b,n,1,w)], dim=-2)
# bottom = torch.cat([torch.zeros(b,n,1,w), weight[:,:,:-1]], dim=-2)
# adj_mat = torch.stack([left+top, right+top, left+bottom, right+bottom], dim=1) # 4bnhw


# Returns DAG mixers
def get_normalized_dag_mixer(expdt, head_node_type, h,w, tol=1e-6): # 2bnhw,
    # 2: corresponds to left_right edges and top_bottom edges
    # b,n,h,w = expdt[0].shape
    # b,n,h,w = expdt.shape
    # t = h*w
    b,n,t = expdt.shape
    adj_mat = torch.zeros(b,n,t,t, device=device)
    # num incident edges on each node
    num_incident_edges = 2*torch.ones(h, w, device=device, requires_grad=False)
    index_set_2d = torch.arange(t, device=device).reshape(h,w)

# index_set_2d:
# 012...w-1
# w...2w-1
# ...
# (h-1)w...hw-1

    # value_set1, value_set2 = expdt.flatten(3) # 2bnhw->(bnt,bnt) # lr,tb
    value_set1 = value_set2 = expdt.flatten(2) # bnhw->(bnt,bnt) # lr,tb
    if head_node_type in ['top_left', 'bottom_left']:
        left_set, right_set = index_set_2d[:,:-1].flatten(), index_set_2d[:,1:].flatten()
        adj_mat[:,:,left_set,right_set] = value_set1[:,:,right_set]
        num_incident_edges[:,0] = num_incident_edges[:,0] - 1
    elif head_node_type in ['top_right', 'bottom_right']:
        left_set, right_set = index_set_2d[:,1:].flatten(), index_set_2d[:,:-1].flatten()
        adj_mat[:,:,left_set,right_set] = value_set1[:,:,right_set]
        num_incident_edges[:,-1] = num_incident_edges[:,-1] - 1
    # else: raise ValueError(f"Invalid head node: {head_node_type}")
    if head_node_type in ['top_left', 'top_right']:
        left_set, right_set = index_set_2d[:-1].flatten(), index_set_2d[1:].flatten()
        adj_mat[:,:,left_set,right_set] = value_set2[:,:,right_set]
        num_incident_edges[0] = num_incident_edges[0] - 1
    elif head_node_type in ['bottom_left', 'bottom_right']:
        left_set, right_set = index_set_2d[1:].flatten(), index_set_2d[:-1].flatten()
        adj_mat[:,:,left_set,right_set] = value_set2[:,:,right_set]
        num_incident_edges[-1] = num_incident_edges[-1] - 1
    # else: raise ValueError(f"Invalid head node: {head_node_type}")

    adj_mat = adj_mat.transpose(-1,-2) # Transpose the last two dimensions, and make it an incoming edges matrix
    num_incident_edges[num_incident_edges < tol] = 1 # Divide each row by the number of non-zero values
    adj_mat = adj_mat / torch.sqrt(num_incident_edges.flatten())[None,None,:,None] # is_normalize T
    return adj_mat, num_incident_edges # bnhw, hw

# expdt = self.unused_dt_mask*torch.exp(-dt) # 4211hw*[num_dt,2,b,n,h,w]
# b,n = 2,3
b,n = 1,1
h,w = 2,2
# expdt = torch.rand(4,2,b,n,h,w, device=device)
expdt = torch.rand(b,n,h*w, device=device)
idx_dag_name = {0: 'top_left', 1: 'top_right', 2: 'bottom_left', 3: 'bottom_right'}
# for dt_idx, dag_name in idx_dag_name.items():
#     A_matrix, num_incident_edges = get_normalized_dag_mixer(expdt[dt_idx], dag_name)
#     break
# dt_idx=0
for dt_idx in range(4):
    # A_matrix, num_incident_edges = get_normalized_dag_mixer(expdt[dt_idx], idx_dag_name[dt_idx])
    A_matrix, num_incident_edges = get_normalized_dag_mixer(expdt, idx_dag_name[dt_idx],h,w)
    print(A_matrix)
# A_matrix, num_incident_edges = get_normalized_dag_mixer(expdt[dt_idx], idx_dag_name[dt_idx])
# print(A_matrix)
# print(A_matrix.shape)
# print(num_incident_edges)
# print(num_incident_edges.shape)
# print(expdt.squeeze().shape)
# print(expdt[dt_idx])
# A:top_left
# 1112
# 0000
# 1000
# 1000
# 0110
# 1<2
# ^ ^
# 3<4

# # top_right
# 1121
# 0100
# 0000
# 1001
# 0100
# # bottom_left
# 1211
# 0010
# 1001
# 0000
# 0010

# # bottom_right
# 2111
# 0110
# 0001
# 0001
# 0000



# num_incident_edges: hw
# top_left
# [[1., 1., 1., 1., 1.],
# [1., 2., 2., 2., 2.],
# [1., 2., 2., 2., 2.],
# [1., 2., 2., 2., 2.],
# [1., 2., 2., 2., 2.]]
# all 2's except two edges adj to corner =1's


In [ ]:
# @title chimera one
# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/src/models/sequence/modules/chimerablock.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# @torch.compile()
class Chimera(nn.Module):
    def __init__(self, d_model, expand=2, n_heads=8, n_groups=8, d_state=8, d_conv=2, height=8, width=8):
        super().__init__()
        n_groups = min(n_heads, n_groups)
        assert n_heads % n_groups == 0, "nheads must be divisible by ngroups"
        self.d_model, self.n_groups, self.d_state, = d_model, n_groups, d_state
        self.d_inner = expand * d_model
        self.n_heads, self.d_head = n_heads, self.d_inner//n_heads
        self.height, self.width = height, width

        self.in_proj = nn.Linear(d_model, 2* self.d_inner + 2* self.n_groups*self.d_state + self.n_heads, bias=False)
        conv_dim = self.d_inner + 2* self.n_groups*self.d_state
        self.conv2d = nn.Conv2d(conv_dim, conv_dim, kernel_size=d_conv*2-1, groups=conv_dim, padding=d_conv-1, bias=True)
        self.act = nn.SiLU()

        # # Mask for directed convolution
        # conv_mask = torch.zeros(d_conv*2-1, d_conv*2-1)
        # conv_mask[0:d_conv, 0:d_conv] = torch.flip(torch.tril(torch.ones(d_conv, d_conv)), dims=[1])
        # conv_mask[0:d_conv, d_conv-1:] = torch.tril(torch.ones(d_conv, d_conv))
        # conv_mask[d_conv-1:, 0:d_conv] = torch.triu(torch.ones(d_conv, d_conv))
        # conv_mask[d_conv-1:, d_conv-1:] = torch.flip(torch.triu(torch.ones(d_conv, d_conv)), dims=[1])
        # nn.init.uniform_(self.conv1d.weight, -conv_init, conv_init)
        # self.conv2d.weight.data *= conv_mask

        dt_min, dt_max = .001, .1
        dt = torch.exp(torch.rand(self.n_heads) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt))) # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        self.dt_bias._no_reinit, self.dt_bias._no_weight_decay = True, True

        self.D = nn.Parameter(torch.ones(self.n_heads, device=device))
        self.D._no_reinit, self.D._no_weight_decay = True, True
        self.norm = nn.RMSNorm(self.d_inner, eps=1e-5)
        # self.dropout = DropoutNd(0, transposed=False) # tie_dropout # from src.models.nn import DropoutNd
        self.dropout = nn.Dropout(0)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        self.num_edge = make_num_edge(height, width) # t

    def forward(self, u): # btd
        t = u.size(-2)
        z, xBC, dt = self.in_proj(u).split([self.d_inner, self.d_inner + 2*self.n_groups*self.d_state, self.n_heads], dim=-1)
        xBC = xBC.transpose(-2,-1).unflatten(-1, (self.height, self.width)) # b(hw)d->bdhw
        xBC = self.act(self.conv2d(xBC))
        xBC = xBC.flatten(2).transpose(1,2) # bdhw->btd
        x, B, C = xBC.split([self.d_inner, self.n_groups*self.d_state, self.n_groups*self.d_state], dim=-1)

        # dt = dt.transpose(-2,-1).unflatten(-1, (self.height, self.width)) # btn->bnhw
        # dt = F.softplus(dt + self.dt_bias[:,None,None]) # bnhw+n11->bnhw # aka ln(Aij)?
        # dA = torch.exp(-dt) # bnhw # ~Abar in mamba
        dt = dt.transpose(-2,-1) # bnt
        dt = F.softplus(dt + self.dt_bias.unsqueeze(-1)) # bnt+n1->bnt # aka ln(Aij)?
        dA = torch.exp(-dt) # bnt # ~Abar in mamba

        x, B, C = x.unflatten(-1, (self.n_heads, self.d_head)), B.unflatten(-1, (self.n_groups, self.d_state)), C.unflatten(-1, (self.n_groups, self.d_state)) # x:[b,t,h,d], B/C:[b,t,g,s]
        h_g = self.n_heads//self.n_groups
        if h_g>1: B, C = B.repeat_interleave(h_g, dim=-2), C.repeat_interleave(h_g, dim=-2) # btgs->bths
        x, B, C = [a.transpose(1,2) for a in (x, B, C)] # X:bhtd, B/C:bhts

        # y = torch.zeros_like(x, device=u.device)
        # idx_dag_name = {0: 'top_left', 1: 'top_right', 2: 'bottom_left', 3: 'bottom_right'}
        # # for dag_name in idx_dag_name.values():
        # for i, dag_name in idx_dag_name.items():
        # A, num_incident_edges = get_normalized_dag_mixer(dA, dag_name) # bnhw->bntt,hw
        num_edge = self.num_edge#[i]
        # print(make_adj_mat(dA, self.height, self.width).shape, num_edge.shape)
        A = make_adj_mat(dA, self.height, self.width) / torch.sqrt(num_edge).unsqueeze(-1) # bntt/t1 # is_normalize T

        # print('A, num_incident_edges', A.shape, num_incident_edges.shape)
        L = DAGInverse.apply(A, self.height+self.width) # use_fast_inverse=T
        # L = torch.inverse(torch.eye(self.height*self.width, device=device) - A) # use_fast_inverse=F

        # sum_dt = torch.sum(dt, dim=0, keepdim=False) # 2bnhw->bnhw # normalization_mode == "dt_original
        # sum_dt = 1-torch.exp(2*torch.sum(dt, dim=0, keepdim=False)) # norm_sqrt_mul_factor=1.0, # < 1 # normalization_mode == "sqrt
        # sum_dt = dt_self # normalization_mode == "dt_self # dt_self = F.softplus(dt_self + self.dt_self_bias; is repeat of dt
        # norm_dt = (sum_dt/torch.sqrt(num_incident_edges)).flatten(2) # bnhw/hw->bnt
        # norm_dt = (dt/torch.sqrt(num_incident_edges)).flatten(2) # bnhw/hw->bnt
        norm_dt = dt/torch.sqrt(num_edge) # bnt/t=bnt
        # print('B, norm_dt', B.shape, norm_dt.shape)
        dB = torch.einsum('bnls,bnl->bnls', B, norm_dt) # [b,n,t,qk_dim]*bnt
        # y = y + torch.einsum('bnls,bnlt,bnts,bntd->bnld', C, L, dB, x)
        y = torch.einsum('bnls,bnlt,bnts,bntd->bnld', C, L, dB, x)
        y = y*4**-.5 #

        y = y + x*self.D[None,:,None,None] # std_dev*y+hid, std_dev=1 This will be per method parameter
        y = y.transpose(1,2).flatten(2) # [b,t,d]/[1,b*t,d]
        y = self.norm(y * F.silu(z)) # self.norm(y) * F.silu(z)
        y = self.out_proj(self.dropout(y))
        return y[:, :t, :], None


d=64
b=3
h,w = 14,14
model = Chimera(d, height=h, width=w).to(device)
x = torch.randn(b, h*w, d, device=device)
y, _ = model(x)
# y = model(x)
print(y.shape)


In [ ]:
# @title chimera
# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/src/models/sequence/modules/chimerablock.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# @torch.compile()
class Chimera(nn.Module):
    # def __init__(self, d_model, expand_factor=2, headdim=64, qk_dim=64, d_conv=2, height=14, width=14): # Need to support dynamic image width
    def __init__(self, d_model, expand_factor=2, headdim=64, qk_dim=64, d_conv=2, height=8, width=8): # Need to support dynamic image width
        super().__init__()
        # print('height, width',height, width)
        self.d_model, self.headdim, self.qk_dim = d_model, headdim, qk_dim
        self.height, self.width = height, width
        self.d_inner = expand_factor*d_model
        assert self.d_inner % self.headdim == 0
        self.num_heads = self.d_inner // self.headdim
        # self.num_b_or_c, self.num_dt = 2, 2
        self.num_b_or_c, self.num_dt = 1, 1

        d_in_proj = self.d_inner + self.d_inner + self.qk_dim*2*self.num_b_or_c + self.num_heads*2*self.num_dt
        self.in_proj = nn.Linear(d_model, d_in_proj, bias=False)
        conv_dim = self.d_inner + self.qk_dim*2*self.num_b_or_c
        self.conv2d = nn.Conv2d(conv_dim, conv_dim, kernel_size=d_conv*2-1, groups=conv_dim, padding=d_conv-1, bias=True)
        self.act = nn.SiLU()

        # # Mask for directed convolution
        # self.conv_mask = torch.zeros(d_conv*2-1, d_conv*2-1)
        # self.conv_mask[0:d_conv, 0:d_conv] = torch.flip(torch.tril(torch.ones(d_conv, d_conv)), dims=[1])
        # self.conv_mask[0:d_conv, d_conv-1:] = torch.tril(torch.ones(d_conv, d_conv))
        # self.conv_mask[d_conv-1:, 0:d_conv] = torch.triu(torch.ones(d_conv, d_conv))
        # self.conv_mask[d_conv-1:, d_conv-1:] = torch.flip(torch.triu(torch.ones(d_conv, d_conv)), dims=[1])
        # nn.init.uniform_(self.conv1d.weight, -conv_init, conv_init)
        # self.conv_mask = self.conv_mask.to(device)
        # self.conv2d.weight.data *= self.conv_mask

        dt_min, dt_max = .001, .1
        dt = torch.exp(torch.rand(self.num_heads) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt))) # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        self.dt_bias._no_reinit, self.dt_bias._no_weight_decay = True, True

        self.unused_dt_mask = get_unused_dt_mask(height, width)[:,:,None,None] #(num_matrix_mixers=[4,2,h,w])->[4,2,1,1,h,w]

        self.D = nn.Parameter(torch.ones(self.num_heads, device=device))
        self.D._no_reinit, self.D._no_weight_decay = True, True
        self.norm = nn.RMSNorm(self.d_inner, eps=1e-5)
        # self.dropout = DropoutNd(0, transposed=False) # tie_dropout # from src.models.nn import DropoutNd
        self.dropout = nn.Dropout(0)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, u): # btd
        t = u.size(-2)
        device = u.device
        z, xBC, dt = self.in_proj(u).split([self.d_inner, self.d_inner + 2* self.num_b_or_c*self.qk_dim, 2*self.num_dt*self.num_heads], dim=-1)
        xBC = xBC.transpose(-2,-1).unflatten(-1, (self.height, self.width)) # b(hw)d->bdhw
        xBC = self.act(self.conv2d(xBC))
        xBC = xBC.flatten(2).transpose(1,2) # bdhw->btd
        x, B, C = xBC.split([self.d_inner, self.num_b_or_c*self.qk_dim, self.num_b_or_c*self.qk_dim], dim=-1)

        x = x.unflatten(-1, (self.num_heads,-1)).transpose(1,2) # bt(nd)->bntd # [b,n,l,qk_dim]

        # Rearrange dt # (i -> rows,j -> column)
        # dt = rearrange(dt, "b (h w) (n p q) -> p q b n h w", h=self.height, w=self.width, n=self.num_heads, p=self.num_dt, q=2)
        dt = rearrange(dt, "b (h w) (p q n) -> p q b n h w", h=self.height, w=self.width, n=self.num_heads, p=self.num_dt, q=2)
        dt = dt.repeat_interleave(4//self.num_dt, dim=0) # graphs_mode line
        # dt = dt.repeat_interleave(2, dim=0) # graphs_mode line
        # dt = torch.cat([dt, dt.flip(0)], dim=0) # graphs_mode diagonal
        # print(self.unused_dt_mask.shape, dt.shape, self.dt_bias.shape)
        dt = self.unused_dt_mask*F.softplus(dt + self.dt_bias[:,None,None]) # 4211hw*([num_dt,2,b,n,h,w]+n11)->4211hw
        expdt = self.unused_dt_mask*torch.exp(-dt) # 4211hw*[num_dt,2,b,n,h,w]=42bnhw # ~Abar in mamba

        B = B.unflatten(-1, (self.num_b_or_c,-1)).permute(2,0,1,3) # btd->[b,t,num_b_or_c,qk_dim]->[num_b_or_c,b,t,qk_dim]
        B = B.unsqueeze(2).repeat(2//self.num_b_or_c,1,self.num_heads,1,1) # [num_b_or_c,b,t,qk_dim]->[num_b_or_c,b,n,t,qk_dim]
        C = C.unflatten(-1, (self.num_b_or_c,-1)).permute(2,0,1,3) # btd->[b,t,num_b_or_c,qk_dim]->[num_b_or_c,b,t,qk_dim]
        C = C.unsqueeze(2).repeat(2//self.num_b_or_c,1,self.num_heads,1,1) # [num_b_or_c,b,t,qk_dim]->[num_b_or_c,b,n,t,qk_dim]

        # Get DAG mixers
        bc_ids = [0,0,1,1] # graphs_mode line:[0,0,1,1]; diagonal:[0,1,1,0]
        # bc_ids = [0,1,1,0]
        y = torch.zeros_like(x, device=device)
        idx_dag_name = {0: 'top_left', 1: 'top_right', 2: 'bottom_left', 3: 'bottom_right'}
        for (dt_idx, dag_name), bc_idx in zip(idx_dag_name.items(), bc_ids):
            A, num_incident_edges = get_normalized_dag_mixer(expdt[dt_idx], dag_name) # 2bnhw->bnhw,hw
            L = DAGInverse.apply(A, self.height+self.width) # use_fast_inverse=T
            # L = torch.inverse(torch.eye(self.height*self.width, device=device) - A) # use_fast_inverse=F

            sum_dt = torch.sum(dt[dt_idx], dim=0, keepdim=False) # 2bnhw->bnhw # normalization_mode == "dt_original
            # sum_dt = 1-torch.exp(2*torch.sum(dt[dt_idx], dim=0, keepdim=False)) # norm_sqrt_mul_factor=1.0, # < 1 # normalization_mode == "sqrt
            # sum_dt = dt_self[dt_idx] # normalization_mode == "dt_self
            norm_dt = (sum_dt/torch.sqrt(num_incident_edges)).flatten(2) # bnhw/hw->bnt

            b_bar = torch.einsum('bnld,bnl->bnld', B[bc_idx], norm_dt) # [b,n,t,qk_dim]*bnt
            y = y + torch.einsum('bnld,bnlt,bntd,bntp->bnlp', C[bc_idx], L, b_bar, x)
        y = y/math.sqrt(self.unused_dt_mask.shape[0])

        y = y + x*self.D[None,:,None,None] # std_dev*y+hid, std_dev=1 This will be per method parameter
        y = y.transpose(1,2).flatten(2) # [b,t,d]/[1,b*t,d]
        y = self.norm(y * F.silu(z)) # self.norm(y) * F.silu(z)
        y = self.dropout(y)
        y = self.out_proj(y)
        return y[:, :t, :], None


d=64
b=3
h,w = 14,14
model = Chimera(d, height=h, width=w).to(device)
x = torch.randn(b, h*w, d, device=device)
y, _ = model(x)
# y = model(x)
print(y.shape)


In [ ]:
# @title chimera down
# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/src/models/sequence/modules/chimera.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
import numpy as np
# from torch_geometric.utils import to_dense_adj
# from torch_geometric.data import Data

class DAGInverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, A, n):
        j = 2**math.ceil(math.log2(n))
        #|A|: (b n l l)
        last_A_j = A
        I_matrix =  torch.eye(A.size(-1), dtype=A.dtype, device=A.device)[None,None,...]
        I_minus_A_inv = I_matrix + A
        # Compute (I+A)(I+A^2)(I+A^4)...(I+A^j)
        # TODO: Do this in log(j) steps, but for j = 32, O(j) is fine
        for _ in range(2, int(math.log2(j)) + 1):
            last_A_j = torch.matmul(last_A_j, last_A_j)
            I_minus_A_inv = torch.matmul(I_minus_A_inv, I_matrix + last_A_j)
        ctx.save_for_backward(I_minus_A_inv)
        return I_minus_A_inv

    @staticmethod
    def backward(ctx, grad_output):
        I_minus_A_inv = ctx.saved_tensors[0] # Retrieve the saved tensors from the forward pass
        grad_A = torch.matmul(I_minus_A_inv.transpose(-2,-1), torch.matmul(grad_output, I_minus_A_inv.transpose(-2,-1))) # Compute the gradient with respect to A
        return grad_A, None

class Chimera(nn.Module):
    def __init__(self, d_model, expand_factor=2, headdim=64, qk_dim=64, height=14, width=14): #Need to support dynamic image width
        super().__init__()
        self.d_model, self.expand_factor, self.headdim, self.qk_dim = d_model, expand_factor, headdim, qk_dim
        self.include_headnodes="1111" # order: top_left, top_right, bottom_left, bottom_right

        self.d_inner = expand_factor * d_model
        assert self.d_inner % self.headdim == 0
        self.num_heads = self.d_inner // self.headdim
        self.tol = 1e-6
        self.height, self.width = height, width

        dt_min, dt_max = .001, .1
        dt = torch.exp(torch.rand(self.num_heads) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt))) # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        self.dt_bias._no_reinit = True
        self.dt_bias._no_weight_decay = True

        # if self.normalization_mode == "dt_self":
        #     self.dt_self_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))
        #     self.dt_self_bias._no_reinit = True
        #     self.dt_self_bias._no_weight_decay = True

        self.D = nn.Parameter(torch.ones(self.num_heads, device=device))
        self.D._no_reinit = True
        self.D._no_weight_decay = True
        self.unused_dt_mask = self.get_unused_dt_mask().unsqueeze(2).unsqueeze(2)

    def get_unused_dt_mask(self):
        # Shape of dt_rearranged: [4,2,b,h,i,j]
        # Must be used in the order [0]top_left, [1]top_right, [2]bottom_left, [3]bottom_right,
        # where each is followed by [0]left-right, [1]top-bottom
        # Visualize a Rectangle with 4 corners
        # 0 --(a)--- 1
        # |          |
        #(d)        (b)
        # |          |
        # 2 --(c)--- 3
        # dt_mask = torch.ones(4, 2, self.height, self.width, requires_grad=False)
        dt_mask = torch.ones(4, 2, self.height, self.width)
        # luruldrd
        # Corners and Edges:
        # For (a) remove [0][1], and [1][1]
        dt_mask[0,1,0,:] = 0
        dt_mask[1,1,0,:] = 0
        # For (b) remove [1][0], and [3][0]
        dt_mask[1,0,:,-1] = 0
        dt_mask[3,0,:,-1] = 0
        # For (c) remove [2][1], and [3][1]
        dt_mask[2,1,-1,:] = 0
        dt_mask[3,1,-1,:] = 0
        # For (d) remove [0][0], and [2][0]
        dt_mask[0,0,:,0] = 0
        dt_mask[2,0,:,0] = 0
        # idxs = [i for i in range(4) if self.include_headnodes[i] == '1'] # [0, 1, 2, 3]
        # idxs = list(range(4)) # [0, 1, 2, 3]
        # print('get_unused_dt_mask',dt_mask)
        return dt_mask#[idxs]

    # Returns DAG mixers
    def get_normalized_dag_mixer_depr(self, expdt, head_node_type, device, is_normalize=True):
        # Shape of dt_rearranged: [2, batch, num_heads, i, j]
        # where (i -> rows,j -> column)
        # 2: corresponds to left_right edges and top_bottom edges
        # |head_node_type|: values top_left, top_right, bottom_left, bottom_right
        # Exponentiate all dts
        height, width = expdt.shape[-2:]
        # num incident edges on each node
        num_incident_edges = 2*torch.ones(height, width, device=device, requires_grad=False)
        # left_right matrix ----------
        # left_right node indices
        lr_node_indices = torch.arange(height*width).reshape(height, width).to(device)
        lr_left_nodes, lr_right_nodes = lr_node_indices[:,:-1].flatten(), lr_node_indices[:,1:].flatten()
        lr_dt = expdt[0] # bnij

        if head_node_type == 'top_left' or head_node_type == 'bottom_left':
            lr_edge_index = torch.stack([lr_left_nodes, lr_right_nodes], dim=0)
            lr_edge_attr = lr_dt[...,1:] # ensures target data dependency
            lr_edge_attr = rearrange(lr_edge_attr, "b n i j -> (i j) b n")
            num_incident_edges[:,0] = num_incident_edges[:,0] - 1
        elif head_node_type == 'top_right' or head_node_type == 'bottom_right':
            lr_edge_index = torch.stack([lr_right_nodes, lr_left_nodes], dim=0)
            lr_edge_attr = lr_dt[...,:-1] # ensures target data dependency
            lr_edge_attr = rearrange(lr_edge_attr, "b n i j -> (i j) b n")
            num_incident_edges[:,-1] = num_incident_edges[:,-1] - 1
        else: raise ValueError(f"Invalid head node: {head_node_type}")

        # top_bottom matrix ----------------
        # top_bottom node indices
        tb_node_indices = torch.arange(height*width).reshape(height, width).to(device)
        tb_top_nodes, lr_right_nodes = tb_node_indices[:-1].flatten(), tb_node_indices[1:].flatten()
        tb_dt = expdt[1] # bnij

        if head_node_type == 'top_left' or head_node_type == 'top_right':
            tb_edge_index = torch.stack([tb_top_nodes, tb_bottom_nodes], dim=0)
            tb_edge_attr = tb_dt[:,:,1:,:]
            tb_edge_attr = rearrange(tb_edge_attr, "b n i j -> (i j) b n")
            num_incident_edges[0,:] = num_incident_edges[0,:] - 1
        elif head_node_type == 'bottom_left' or head_node_type == 'bottom_right':
            tb_edge_index = torch.stack([tb_bottom_nodes, tb_top_nodes], dim=0)
            tb_edge_attr = tb_dt[:,:,:-1,:]
            tb_edge_attr = rearrange(tb_edge_attr, "b n i j -> (i j) b n")
            num_incident_edges[-1,:] = num_incident_edges[-1,:] - 1
        else: raise ValueError(f"Invalid head node: {head_node_type}")

        print(lr_edge_index, tb_edge_index)
        print(lr_edge_attr, tb_edge_attr)
        graph_data = Data(edge_index=torch.concatenate((lr_edge_index, tb_edge_index), dim=1),
            edge_attr=torch.concatenate((lr_edge_attr, tb_edge_attr), dim=0))
        adj_mat = to_dense_adj(graph_data.edge_index, edge_attr=graph_data.edge_attr)

        # adj_mat = rearrange(adj_mat[0], "l t b n -> b n l t").transpose(-1, -2)
        adj_mat = rearrange(adj_mat[0], "l t b n -> b n t l")# incoming edges matrix
         # Divide each row by the number of non-zero values
        num_incident_edges[num_incident_edges < self.tol] = 1
        if is_normalize: adj_mat = adj_mat / torch.sqrt(rearrange(num_incident_edges, "i j -> (i j)"))[None,None,:,None]
        return adj_mat, num_incident_edges

    # Returns DAG mixers
    def get_normalized_dag_mixer(self, expdt, head_node_type, device, is_normalize=True):
        # Shape of dt_rearranged: [2, batch, num_heads, i, j]
        # where (i -> rows,j -> column)
        # 2: corresponds to left_right edges and top_bottom edges
        # |head_node_type|: values top_left, top_right, bottom_left, bottom_right

        batch, num_heads, height, width = expdt[0].shape
        seq_len = height * width

        adj_mat = torch.zeros(batch, num_heads, seq_len, seq_len, device=device)
        # num incident edges on each node
        num_incident_edges = 2*torch.ones(height, width, device=device, requires_grad=False)

        # left_right matrix ----------
        index_set_2d = torch.arange(seq_len).reshape(height, width).to(device)

        # left_right node dt values
        lr_dt = expdt[0] # bnij
        tb_dt = expdt[1] # bnij

        # if head_node_type == 'top_left' or head_node_type == 'bottom_left':
        if head_node_type in ['top_left', 'bottom_left']:
            left_set = index_set_2d[:,:-1].flatten()
            right_set = index_set_2d[:,1:].flatten()
            value_set = rearrange(lr_dt, "b n i j -> b n (i j)")
            adj_mat[:,:,left_set,right_set] = value_set[:,:,right_set]
            num_incident_edges[:,0] = num_incident_edges[:,0] - 1
        # elif head_node_type == 'top_right' or head_node_type == 'bottom_right':
        elif head_node_type in ['top_right', 'bottom_right']:
            left_set = index_set_2d[:,1:].flatten()
            right_set = index_set_2d[:,:-1].flatten()
            value_set = rearrange(lr_dt, "b n i j -> b n (i j)")
            adj_mat[:,:,left_set,right_set] = value_set[:,:,right_set]
            num_incident_edges[:,-1] = num_incident_edges[:,-1] - 1
        # else: raise ValueError(f"Invalid head node: {head_node_type}")

        # # top_bottom matrix ----------------
        # top_bottom node dt values

        # if head_node_type == 'top_left' or head_node_type == 'top_right':
        if head_node_type in ['top_left', 'top_right']:
            left_set = index_set_2d[:-1,:].flatten()
            right_set = index_set_2d[1:,:].flatten()
            value_set = rearrange(tb_dt, "b n i j -> b n (i j)")
            adj_mat[:,:,left_set,right_set] = value_set[:,:,right_set]
            num_incident_edges[0,:] = num_incident_edges[0,:] - 1
        # elif head_node_type == 'bottom_left' or head_node_type == 'bottom_right':
        elif head_node_type in ['bottom_left', 'bottom_right']:
            left_set = index_set_2d[1:,:].flatten()
            right_set = index_set_2d[:-1,:].flatten()
            value_set = rearrange(tb_dt, "b n i j -> b n (i j)")
            adj_mat[:,:,left_set,right_set] = value_set[:,:,right_set]
            num_incident_edges[-1,:] = num_incident_edges[-1,:] - 1
        else: raise ValueError(f"Invalid head node: {head_node_type}")

        adj_mat = adj_mat.transpose(-1,-2) # Transpose the last two dimensions, and make it an incoming edges matrix
        num_incident_edges[num_incident_edges < self.tol] = 1 # Divide each row by the number of non-zero values

        if is_normalize: adj_mat = adj_mat / torch.sqrt(rearrange(num_incident_edges, "i j -> (i j)"))[None,None,:,None]
        return adj_mat, num_incident_edges


    # def forward(self, x, wgt_params_data, bc, dt_self=None): # btd
    def forward(self, x, dt, bc): # btd
        assert self.height*self.width == x.shape[1]
        batch_size = x.shape[0]
        num_matrix_mixers = 4
        device = x.device
        x = rearrange(x, 'b l (n h) -> b n l h', n=self.num_heads) # [b,h,l,qk_dim]
        self.unused_dt_mask = self.unused_dt_mask.to(device) #(num_matrix_mixers,2,i,j)

        # get_normalized_dag_mixer = self.get_normalized_dag_mixer_depr
        get_normalized_dag_mixer = self.get_normalized_dag_mixer # debug_use_get_A_dpr F

        # Rearrange dt # (i -> rows,j -> column)
        self.num_dt = 2
        dt = rearrange(dt, "b (h w) (n p q) -> p q b n h w", h=self.height, w=self.width, n=self.num_heads, p=self.num_dt, q=2)
        dt = dt.repeat_interleave(2, dim=0) # graphs_mode line
        # dt = torch.cat([dt, dt.flip(0)], dim=0) # graphs_mode diagonal
        dt = self.unused_dt_mask*F.softplus(dt + self.dt_bias[:,None,None])
        # print(self.unused_dt_mask.shape, dt.shape, self.dt_bias.shape)
        expdt = self.unused_dt_mask*torch.exp(-dt)
        # torch.Size([4, 2, 1, 1, 14, 14]) torch.Size([8, 2, 1, 2, 14, 14]) torch.Size([2])


        # normalization_mode="dt_original", # "dt_self", "sqrt"
        # if self.normalization_mode == "dt_self":
        #     dt_self = rearrange(dt_self, "b (i j) (n p) -> p b n i j", i=self.height, j=self.width, n=self.num_heads, p=4)
        #     dt_self = F.softplus(dt_self + self.dt_self_bias[...,None,None])

        self.num_b_or_c = 2

        b,c = bc
        b = b.unsqueeze(1).repeat(1,self.num_heads,1,1) # shape: b n l d
        c = c.unsqueeze(1).repeat(1,self.num_heads,1,1) # shape: b n l d
        b = rearrange(b, 'b n l (m d) -> m b n l d', m=self.num_b_or_c)
        c = rearrange(c, 'b n l (m d) -> m b n l d', m=self.num_b_or_c)

        idx_dag_name = {0: 'top_left', 1: 'top_right', 2: 'bottom_left', 3: 'bottom_right'}
        dt_idx = 0 # serves as idx for both dt and dt_self
        I_matrix = torch.eye(self.height*self.width, device=device)
        # if self.unified_view == False:
        # Get DAG mixers
        bc_idx = 0
        y = torch.zeros_like(x, device=device)
        for key in idx_dag_name.keys():
            if self.include_headnodes[key] == '1':
                A_matrix, num_incident_edges = get_normalized_dag_mixer(expdt[dt_idx], idx_dag_name[key], device, True)
                matrix_mixer = DAGInverse.apply(A_matrix.to(torch.float32), self.height+self.width) # use_fast_inverse=T
                # matrix_mixer = torch.inverse(I_matrix - A_matrix) # use_fast_inverse=F

                summed_dt = torch.sum(dt[dt_idx], dim=0, keepdim=False) # [B,N,i,j] # normalization_mode == "dt_original
                # summed_dt = 1-torch.exp(2*torch.sum(dt[dt_idx], dim=0, keepdim=False)) # norm_sqrt_mul_factor=1.0, # < 1 # normalization_mode == "sqrt
                # summed_dt = dt_self[dt_idx] # normalization_mode == "dt_self
                norm_dt = summed_dt/torch.sqrt(num_incident_edges) #

                rearrange_norm_dt = rearrange(norm_dt, 'b n i j -> b n (i j)')
                b_bar = torch.einsum('bnld,bnl->bnld', b[bc_idx], rearrange_norm_dt)
                y = y + torch.einsum('bnld,bnlt,bntd,bnth->bnlh', c[bc_idx], matrix_mixer, b_bar, x)
                bc_idx += (int(key)%2 == 1) # graphs_mode line
                # bc_idx = bc_idx if int(key)%2 == 1 else (bc_idx + 1)%2 # graphs_mode diagonal
                dt_idx += 1
        y = y/np.sqrt(num_matrix_mixers)
        # y = y/np.sqrt(self.unused_dt_mask.shape[0])
        # else:
        #     A_matrix = torch.zeros_like(I_matrix, device=device)
        #     for key in idx_dag_name.keys():
        #         if self.include_headnodes[key] == '1': # T
        #             A_matrix_temp, num_incident_edges = get_normalized_dag_mixer(expdt[dt_idx], idx_dag_name[key], device, True)
        #             A_matrix = A_matrix + A_matrix_temp
        #             dt_idx += 1
        #     IA_matrix = I_matrix - A_matrix
        #     # if self.normalization_mode == "dt_self":
        #     #     leq = rearrange(dt_self[0], 'b n i j -> b n (i j)')
        #     #     row_sum = torch.sum(IA_matrix, dim=-1, keepdim=False) + leq
        #     #     IA_matrix = .95*IA_matrix/row_sum.unsqueeze(-1) # inverse_factor=0.95
        #     y = torch.einsum('bnld,bnlt,bntd,bnth->bnlh', c[0], torch.inverse(IA_matrix), b[0], x)

        y = y + x*self.D[None,:,None,None] # std_dev*y+hid, std_dev=1 This will be per method parameter
        y = y.transpose(1,2).flatten(2) # [b,t,d]/[1,b*t,d]
        return y



In [ ]:
# @title chimerablock down
# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/src/models/sequence/modules/chimerablock.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
# from .chimera import Chimera
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class ChimeraBlock(nn.Module):
    def __init__(self, d_model, expand_factor=2, headdim=64, qk_dim=64, d_conv=2, height=14, width=14): # Need to support dynamic image width
        super().__init__()
        self.headdim, self.qk_dim = headdim, qk_dim
        self.height, self.width = height, width
        self.d_inner = expand_factor*d_model
        assert self.d_inner % self.headdim == 0
        self.num_heads = self.d_inner // self.headdim

        self.num_b_or_c = 2
        self.num_dt = 2
        self.num_dt_self = 4
        # self.is_add_dt_self = self.normalization_mode == "dt_self" # F

        self.mixer = Chimera(d_model)
        # d_in_proj = self.d_inner + self.d_inner + self.qk_dim*2*self.num_b_or_c + self.num_heads*2*self.num_dt + self.num_heads*self.num_dt_self*self.is_add_dt_self
        d_in_proj = self.d_inner + self.d_inner + self.qk_dim*2*self.num_b_or_c + self.num_heads*2*self.num_dt
        self.in_proj = nn.Linear(d_model, d_in_proj, bias=False)
        conv_dim = self.d_inner + self.qk_dim*2*self.num_b_or_c
        self.conv2d = nn.Conv2d(conv_dim, conv_dim, kernel_size=d_conv*2-1, groups=conv_dim, padding=d_conv-1, bias=True)
        self.act = nn.SiLU()
        # # Mask for directed convolution
        # self.conv_mask = torch.zeros(d_conv*2-1, d_conv*2-1)
        # self.conv_mask[0:d_conv, 0:d_conv] = torch.flip(torch.tril(torch.ones(d_conv, d_conv)), dims=[1])
        # self.conv_mask[0:d_conv, d_conv-1:] = torch.tril(torch.ones(d_conv, d_conv))
        # self.conv_mask[d_conv-1:, 0:d_conv] = torch.triu(torch.ones(d_conv, d_conv))
        # self.conv_mask[d_conv-1:, d_conv-1:] = torch.flip(torch.triu(torch.ones(d_conv, d_conv)), dims=[1])

        # nn.init.uniform_(self.conv1d.weight, -conv_init, conv_init)

        self.norm = nn.RMSNorm(self.d_inner, eps=1e-5)
        # self.dropout = DropoutNd(0, transposed=False) # tie_dropout # from src.models.nn import DropoutNd
        self.dropout = nn.Dropout(0)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        # if self.add_fc_layers: # F
        #     ff_exp = 4
        #     self.fc_blocks = nn.Sequential(
        #         nn.Linear(d_model, ff_exp*d_model), nn.SiLU(),
        #         nn.Linear(ff_exp*d_model, d_model), nn.Dropout(dropout)
        #     )
            # self.fc_layer_norm = nn.LayerNorm(d_model)

    def forward(self, u, state=None, bias=None, **kwargs): # btd
        L = u.size(-2)
        device = u.device

        zxBCdt = self.in_proj(u)
        # if self.normalization_mode == "dt_self":
        #     z, xBC, dt, dt_self = zxBCdt.split([self.d_inner, self.d_inner + self.qk_dim*2*self.num_b_or_c, self.num_heads*2*self.num_dt, self.num_heads*self.num_dt_self], dim=-1)
        # else:
        z, xBC, dt = zxBCdt.split([self.d_inner, self.d_inner + (self.qk_dim*2*self.num_b_or_c), self.num_heads*2*self.num_dt], dim=-1)
        dt_self = None

        # shape of |xBC| is "b (i j) d"
        xBC = rearrange(xBC, 'b (i j) d -> b d i j', i=self.height, j=self.width)
        # self.conv_mask = self.conv_mask.to(device)
        # self.conv2d.weight.data *= self.conv_mask
        xBC = self.act(self.conv2d(xBC))
        xBC = rearrange(xBC, 'b d i j-> b (i j) d', i=self.height, j=self.width)
        x, B, C = xBC.split([self.d_inner, self.num_b_or_c*self.qk_dim, self.num_b_or_c*self.qk_dim], dim=-1)

        # y = self.mixer(x, dt, [B,C], dt_self)
        y = self.mixer(x, dt, [B,C])
        y = self.norm(y) * F.silu(z)
        y = self.dropout(y)

        # if not torch.is_autocast_enabled(): y = y.to(dtype=self.out_proj.weight.dtype) #y could be in fp32 because of the SSMs
        y = self.out_proj(y)
        y = y[:, :L, :]

        # if self.add_fc_layers: # F
        #     residual = y
        #     y = self.fc_blocks(y)
        #     y = self.fc_layer_norm(y + residual)
        return y, None


d=64
b=3
h,w = 14,14
model = ChimeraBlock(d, height=h, width=w)
x = torch.randn(b, h*w, d)
y, _ = model(x)
print(y.shape)


In [ ]:
# @title goombalab/language_experiments/chimerablock.py save
# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/src/models/sequence/modules/chimerablock.py
import math
from functools import partial
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
from src.models.nn import DropoutNd
from mamba_ssm.ops.triton.layernorm import RMSNorm
# from .chimera import Chimera

class ChimeraBlock(nn.Module):
    def __init__(
        self,
        d_model,
        qk_dim=64,
        headdim=64,

        include_headnodes="1111", # order: top_left, top_right, bottom_left, bottom_right
        # other configs
        d_conv=2,
        bias=False,
        dropout=0.0,
        tie_dropout=False,
        image_height=14, #Need to support dynamic image height
        image_width=14, #Need to support dynamic image width
    ):
        super().__init__()
        self.d_model = d_model
        self.qk_dim = qk_dim
        self.headdim= headdim

        # graph mamba flags
        self.include_headnodes = include_headnodes

        self.bias = bias
        self.image_height = image_height
        self.image_width = image_width

        dropout_fn = partial(DropoutNd, transposed=False) if tie_dropout else nn.Dropout
        self.dropout = dropout_fn(dropout) if dropout > 0.0 else nn.Identity()

        self.d_inner = expand_factor*d_model


        assert self.d_inner % self.headdim == 0
        self.num_heads = self.d_inner // self.headdim

        # Build the sequence mixer
        self.mixer = Chimera(
            d_model=self.d_model,
            qk_dim=self.qk_dim,
            headdim=self.headdim,
            include_headnodes=self.include_headnodes,
            image_height=self.image_height,
            image_width=self.image_width)
        self.act = nn.SiLU()

        # self.num_b_or_c = 1 if self.share_BC or self.unified_view else sum(map(int, self.include_headnodes))
        self.num_b_or_c = sum(map(int, self.include_headnodes))
        assert sum(map(int, self.include_headnodes)) == 4
        self.num_b_or_c = self.num_b_or_c - 2

        self.num_dt = sum(map(int, self.include_headnodes))
        self.num_dt = self.num_dt - 2

        # self.num_dt_self = 1 if self.unified_view else sum(map(int, self.include_headnodes))
        self.num_dt_self = sum(map(int, self.include_headnodes))
        self.is_add_dt_self = self.normalization_mode == "dt_self"

        d_in_proj = self.d_inner + self.d_inner + self.qk_dim*2*self.num_b_or_c + self.num_heads*2*self.num_dt + self.num_heads*self.num_dt_self*self.is_add_dt_self
        conv_dim = self.d_inner + self.qk_dim*2*self.num_b_or_c

        self.in_proj = nn.Linear(self.d_model, d_in_proj, bias=False)
        self.conv2d = nn.Conv2d(conv_dim, conv_dim, kernel_size=self.d_conv*2-1, groups=conv_dim, padding=self.d_conv-1, bias=True)
        # # Mask for directed convolution
        # self.directed_conv_mask = torch.zeros((self.d_conv * 2 - 1, self.d_conv * 2 - 1))
        # if self.include_headnodes[0] == '1':
        #     self.directed_conv_mask[0: self.d_conv, 0: self.d_conv] = torch.flip(torch.tril(
        #         torch.ones(self.d_conv, self.d_conv)), dims=[1])
        # if self.include_headnodes[1] == '1':
        #     self.directed_conv_mask[0: self.d_conv, self.d_conv - 1:] = torch.tril(
        #         torch.ones(self.d_conv, self.d_conv))
        # if self.include_headnodes[2] == '1':
        #     self.directed_conv_mask[self.d_conv - 1:, 0: self.d_conv] = torch.triu(
        #         torch.ones(self.d_conv, self.d_conv))
        # if self.include_headnodes[3] == '1':
        #     self.directed_conv_mask[self.d_conv - 1:, self.d_conv - 1:] = torch.flip(torch.triu(
        #         torch.ones(self.d_conv, self.d_conv)), dims=[1])

        # nn.init.uniform_(self.conv1d.weight, -conv_init, conv_init)

        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=bias)
        self.norm = RMSNorm(self.d_inner, eps=1e-5)

        # if self.add_fc_layers: # F
        #     ff_exp = 4
        #     self.fc_blocks = nn.Sequential(
        #         nn.Linear(d_model, ff_exp*d_model), nn.SiLU(),
        #         nn.Linear(ff_exp*d_model, d_model), nn.Dropout(dropout)
        #     )
            # self.fc_layer_norm = nn.LayerNorm(d_model)

    def forward(self, u, state=None, bias=None, **kwargs): # btd
        L = u.size(-2)
        device = u.device

        zxBCdt = self.in_proj(u)
        if self.normalization_mode == "dt_self":
            z, xBC, dt, dt_self = zxBCdt.split([self.d_inner, self.d_inner + self.qk_dim*2*self.num_b_or_c, self.num_heads*2*self.num_dt, self.num_heads*self.num_dt_self], dim=-1)
        else:
            z, xBC, dt = zxBCdt.split([self.d_inner, self.d_inner + (self.qk_dim*2*self.num_b_or_c), self.num_heads*2*self.num_dt], dim=-1)
            dt_self = None

        # shape of |xBC| is "b (i j) d"
        xBC = rearrange(xBC, 'b (i j) d -> b d i j', i=self.image_height, j=self.image_width)
        # self.directed_conv_mask = self.directed_conv_mask.to(device)
        # self.conv2d.weight.data *= self.directed_conv_mask
        xBC = self.act(self.conv2d(xBC))
        xBC = rearrange(xBC, 'b d i j-> b (i j) d', i=self.image_height, j=self.image_width)
        x, B, C = xBC.split([self.d_inner, self.num_b_or_c*self.qk_dim, self.num_b_or_c*self.qk_dim], dim=-1)

        y = self.mixer(x, dt, [B,C], dt_self)
        y = self.norm(y, z) # y = rmsnorm(y) * F.silu(z)
        y = self.dropout(y)

        #y could be in fp32 because of the SSMs
        if not torch.is_autocast_enabled():
            y = y.to(dtype=self.out_proj.weight.dtype)
        y = self.out_proj(y)
        y = y[:, :L, :]

        # if self.add_fc_layers: # F
        #     residual = y
        #     y = self.fc_blocks(y)
        #     y = self.fc_layer_norm(y + residual)
        return y, None


In [ ]:
# @title chimera base
# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/src/models/sequence/modules/chimera.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
import numpy as np
import sys
from torch_geometric.utils import to_dense_adj
from torch_geometric.data import Data

class DAGInverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, A, n):
        j = 2**math.ceil(math.log2(n))
        #|A|: (b n l l)
        last_A_j = A
        I_matrix =  torch.eye(A.size(-1), dtype=A.dtype, device=A.device).unsqueeze(0).unsqueeze(0)
        I_minus_A_inv = I_matrix + A
        # Compute (I+A)(I+A^2)(I+A^4)...(I+A^j)
        # TODO: Do this in log(j) steps, but for j = 32, O(j) is fine
        for _ in range(2, int(math.log2(j)) + 1):
            last_A_j = torch.matmul(last_A_j, last_A_j)
            I_minus_A_inv = torch.matmul(I_minus_A_inv, I_matrix + last_A_j)
        ctx.save_for_backward(I_minus_A_inv)
        return I_minus_A_inv

    @staticmethod
    def backward(ctx, grad_output):
        I_minus_A_inv = ctx.saved_tensors[0] # Retrieve the saved tensors from the forward pass
        grad_A = torch.matmul(I_minus_A_inv.transpose(-2, -1), torch.matmul(grad_output, I_minus_A_inv.transpose(-2, -1))) # Compute the gradient with respect to A
        return grad_A, None

class Chimera(nn.Module):
    def __init__(self, d_model, expand_factor=2, headdim=64, qk_dim=64,
        include_headnodes="1111", # order: top_left, top_right, bottom_left, bottom_right
        debug_use_get_A_dpr=False,
        height=14, image_width=14): #Need to support dynamic image width
        super().__init__()
        self.d_model, self.expand_factor, self.headdim, self.qk_dim = d_model, expand_factor, headdim, qk_dim

        # graph mamba flags
        self.unified_view = False
        self.include_headnodes = include_headnodes
        self.debug_use_get_A_dpr = debug_use_get_A_dpr

        self.d_inner = round(self.expand_factor * self.d_model)
        assert self.d_inner % self.headdim == 0
        self.num_heads = self.d_inner // self.headdim
        self.std_dev = 1.0 # This will be per method parameter
        self.tol = 1e-6
        self.height, self.image_width = height, image_width

        dt_min, dt_max = .001, .1
        dt = torch.exp(torch.rand(self.num_heads) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt))) # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        self.dt_bias._no_reinit = True
        self.dt_bias._no_weight_decay = True

        # if self.normalization_mode == "dt_self":
        #     dt_self = torch.exp(torch.rand(self.num_heads) * (math.log(self.dt_self_max) - math.log(self.dt_self_min)) + math.log(self.dt_self_min))
        #     dt_self = torch.clamp(dt_self, min=1e-4)
        #     inv_dt_self = dt_self + torch.log(-torch.expm1(-dt_self))
        #     self.dt_self_bias = nn.Parameter(inv_dt_self)
        #     self.dt_self_bias._no_reinit = True
        #     self.dt_self_bias._no_weight_decay = True

        # D "skip" parameter
        self.D = nn.Parameter(torch.ones(self.num_heads, device=device))
        self.D._no_reinit = True
        self.D._no_weight_decay = True
        self.unused_dt_mask = self.get_unused_dt_mask().unsqueeze(2).unsqueeze(2)

    def get_unused_dt_mask(self):
        # Shape of dt_rearranged: [4,2,b,h,i,j]
        # Must be used in the order [0]top_left, [1]top_right, [2]bottom_left, [3]bottom_right,
        # where each is followed by [0]left-right, [1]top-bottom

        # Visualize a Rectangle with 4 corners
        # 0 --(a)--- 1
        # |          |
        #(d)        (b)
        # |          |
        # 2 --(c)--- 3
        dt_mask = torch.ones((4, 2, self.height, self.image_width), requires_grad=False)
        # Corners and Edges:
        # For (a) remove [0][1], and [1][1]
        dt_mask[0][1][0, :] = 0
        dt_mask[1][1][0, :] = 0
        # For (b) remove [1][0], and [3][0]
        dt_mask[1][0][:, -1] = 0
        dt_mask[3][0][:, -1] = 0
        # For (c) remove [2][1], and [3][1]
        dt_mask[2][1][-1, :] = 0
        dt_mask[3][1][-1, :] = 0
        # For (d) remove [0][0], and [2][0]
        dt_mask[0][0][:, 0] = 0
        dt_mask[2][0][:, 0] = 0
        idxs = [i for i in range(4) if self.include_headnodes[i] == '1']
        return dt_mask[idxs]

    # Returns DAG mixers
    def get_normalized_dag_mixer_depr(self,  expdt,  head_node_type,  device, is_normalize=True):
        # Shape of dt_rearranged: [2, batch, num_heads, i, j]
        # where (i -> rows,j -> column)
        # 2: corresponds to left_right edges and top_bottom edges
        # |head_node_type|: values top_left, top_right, bottom_left, bottom_right
        # Exponentiate all dts
        height, width = expdt.shape[-2:]
        # num incident edges on each node
        num_incident_edges = 2*torch.ones((height, width), device=device, requires_grad=False)
        # left_right matrix ----------
        # left_right node indices
        lr_node_indices = torch.arange(height*width).reshape(height, width).to(device)

        lr_left_nodes = lr_node_indices[:, :-1].flatten()
        lr_right_nodes = lr_node_indices[:, 1:].flatten()
        lr_dt = expdt[0] # (b n i j)

        if head_node_type == 'top_left' or head_node_type == 'bottom_left':
            lr_edge_index = torch.stack([lr_left_nodes, lr_right_nodes], dim=0)
            lr_edge_attr = lr_dt[:,:,:,1:] # ensures target data dependency
            lr_edge_attr = rearrange(lr_edge_attr, "b n i j -> (i j) b n")
            num_incident_edges[:,0] = num_incident_edges[:,0] - 1
        elif head_node_type == 'top_right' or head_node_type == 'bottom_right':
            lr_edge_index = torch.stack([lr_right_nodes, lr_left_nodes], dim=0)
            lr_edge_attr = lr_dt[:,:,:,:-1] # ensures target data dependency
            lr_edge_attr = rearrange(lr_edge_attr, "b n i j -> (i j) b n")
            num_incident_edges[:,-1] = num_incident_edges[:,-1] - 1
        else: raise ValueError(f"Invalid head node: {head_node_type}")

        # top_bottom matrix ----------------
        # top_bottom node indices
        tb_node_indices = torch.arange(height*width).reshape(height, width).to(device)

        tb_top_nodes = tb_node_indices[:-1, :].flatten()
        tb_bottom_nodes = tb_node_indices[1:, :].flatten()
        tb_dt = expdt[1] # (b n i j)

        if head_node_type == 'top_left' or head_node_type == 'top_right':
            tb_edge_index = torch.stack([tb_top_nodes, tb_bottom_nodes], dim=0)
            tb_edge_attr = tb_dt[:,:,1:,:]
            tb_edge_attr = rearrange(tb_edge_attr, "b n i j -> (i j) b n")
            num_incident_edges[0,:] = num_incident_edges[0,:] - 1
        elif head_node_type == 'bottom_left' or head_node_type == 'bottom_right':
            tb_edge_index = torch.stack([tb_bottom_nodes, tb_top_nodes], dim=0)
            tb_edge_attr = tb_dt[:,:,:-1,:]
            tb_edge_attr = rearrange(tb_edge_attr, "b n i j -> (i j) b n")
            num_incident_edges[-1,:] = num_incident_edges[-1,:] - 1
        else: raise ValueError(f"Invalid head node: {head_node_type}")

        graph_data = Data(edge_index=torch.concatenate((lr_edge_index, tb_edge_index), dim=1),
            edge_attr=torch.concatenate((lr_edge_attr, tb_edge_attr), dim=0))

        adjacency_matrix = to_dense_adj(graph_data.edge_index, edge_attr=graph_data.edge_attr)
        adjacency_matrix = rearrange(adjacency_matrix[0], "l t b n -> b n l t")

        # Transpose the last two dimensions, and make it an incoming edges matrix
        adjacency_matrix = adjacency_matrix.transpose(-1, -2)
         # Divide each row by the number of non-zero values
        num_incident_edges[num_incident_edges < self.tol] = 1
        if is_normalize: final_adjacency_matrix = adjacency_matrix / torch.sqrt(rearrange(num_incident_edges, "i j -> (i j)"))[None,None,:,None]
        else: final_adjacency_matrix = adjacency_matrix
        return final_adjacency_matrix, num_incident_edges

    # Returns DAG mixers
    def get_normalized_dag_mixer(self, expdt, head_node_type, device, is_normalize=True):
        # Shape of dt_rearranged: [2, batch, num_heads, i, j]
        # where (i -> rows,j -> column)
        # 2: corresponds to left_right edges and top_bottom edges
        # |head_node_type|: values top_left, top_right, bottom_left, bottom_right

        batch, num_heads, height, width = expdt[0].shape
        seq_len = height * width

        # Matrix of all zeros (b n i j l)
        adjacency_matrix = torch.zeros((batch, num_heads, height*width, height*width), device=device)
        # num incident edges on each node
        num_incident_edges = 2*torch.ones((height, width), device=device, requires_grad=False)

        # left_right matrix ----------
        index_set_2d = torch.arange(0, seq_len).reshape(height, width).to(device)

        # left_right node dt values
        lr_dt = expdt[0] # (b n i j)

        if head_node_type == 'top_left' or head_node_type == 'bottom_left':
            left_set = index_set_2d[:, :-1].flatten()
            right_set = index_set_2d[:, 1:].flatten()
            value_set = rearrange(lr_dt, "b n i j -> b n (i j)")
            adjacency_matrix[:, :, left_set, right_set] = value_set[:, :, right_set]
            num_incident_edges[:,0] = num_incident_edges[:,0] - 1
        elif head_node_type == 'top_right' or head_node_type == 'bottom_right':
            left_set = index_set_2d[:, 1:].flatten()
            right_set = index_set_2d[:, :-1].flatten()
            value_set = rearrange(lr_dt, "b n i j -> b n (i j)")
            adjacency_matrix[:, :, left_set, right_set] = value_set[:, :, right_set]
            num_incident_edges[:,-1] = num_incident_edges[:,-1] - 1
        else: raise ValueError(f"Invalid head node: {head_node_type}")

        # # top_bottom matrix ----------------
        # top_bottom node dt values
        tb_dt = expdt[1] # (b n i j)

        if head_node_type == 'top_left' or head_node_type == 'top_right':
            left_set = index_set_2d[:-1, :].flatten()
            right_set = index_set_2d[1:, :].flatten()
            value_set = rearrange(tb_dt, "b n i j -> b n (i j)")
            adjacency_matrix[:, :, left_set, right_set] = value_set[:, :, right_set]
            num_incident_edges[0,:] = num_incident_edges[0,:] - 1
        elif head_node_type == 'bottom_left' or head_node_type == 'bottom_right':
            left_set = index_set_2d[1:, :].flatten()
            right_set = index_set_2d[:-1, :].flatten()
            value_set = rearrange(tb_dt, "b n i j -> b n (i j)")
            adjacency_matrix[:, :, left_set, right_set] = value_set[:, :, right_set]
            num_incident_edges[-1,:] = num_incident_edges[-1,:] - 1
        else: raise ValueError(f"Invalid head node: {head_node_type}")

        adjacency_matrix = adjacency_matrix.transpose(-1,-2) # Transpose the last two dimensions, and make it an incoming edges matrix
        num_incident_edges[num_incident_edges < self.tol] = 1 # Divide each row by the number of non-zero values

        if is_normalize: final_adjacency_matrix = adjacency_matrix / torch.sqrt(rearrange(num_incident_edges, "i j -> (i j)"))[None,None,:,None]
        else: final_adjacency_matrix = adjacency_matrix
        return final_adjacency_matrix, num_incident_edges


    def forward(self, hidden_states, wgt_params_data, bc, dt_self=None):
        assert self.height*self.image_width == hidden_states.shape[1]
        batch_size = hidden_states.shape[0]
        num_matrix_mixers = sum(map(int, self.include_headnodes))
        hidden_states_rearranged = rearrange(hidden_states, 'b l (n h) -> b n l h', n=self.num_heads) # [b,h,l,qk_dim]
        device = hidden_states.device
        self.unused_dt_mask = self.unused_dt_mask.to(device) #(num_matrix_mixers,2,i,j)

        if self.debug_use_get_A_dpr: get_normalized_dag_mixer = self.get_normalized_dag_mixer_depr
        else: get_normalized_dag_mixer = self.get_normalized_dag_mixer

        # Rearrange dt
        # (i -> rows,j -> column)

        # Rearrange b and c
        self.num_dt = sum(map(int, self.include_headnodes))
        self.num_dt = self.num_dt - 2
        dt = rearrange(wgt_params_data,  "b (i j) (n p q) -> p q b n i j",  i=self.height, j=self.image_width, n=self.num_heads, p=self.num_dt, q=2)

        dt = torch.repeat_interleave(dt, 2, dim=0) # graphs_mode line
        dt = torch.cat([dt, dt.flip(0)], dim=0) # graphs_mode diagonal

        dt = dt + self.dt_bias.unsqueeze(-1).unsqueeze(-1)
        dt = self.unused_dt_mask*F.softplus(dt)
        expdt = self.unused_dt_mask*torch.exp(-1*dt)

        normalization_mode="dt_original", # "dt_self", "sqrt"
        # if self.normalization_mode == "dt_self":
        #     dt_self = rearrange(dt_self, "b (i j) (n p) -> p b n i j", i=self.height, j=self.image_width,
        #         n=self.num_heads, p=1 if self.unified_view else sum(map(int, self.include_headnodes)))
        #     dt_self = F.softplus(dt_self + self.dt_self_bias[...,None,None])

        # Rearrange b and c
        # self.num_b_or_c = 1 if self.share_BC or self.unified_view else sum(map(int, self.include_headnodes))
        self.num_b_or_c = sum(map(int, self.include_headnodes))

        assert  sum(map(int, self.include_headnodes)) == 4
        self.num_b_or_c = self.num_b_or_c - 2

        b,c = bc
        b = b.unsqueeze(1).repeat(1, self.num_heads, 1, 1) # shape: b n l d
        c = c.unsqueeze(1).repeat(1, self.num_heads, 1, 1) # shape: b n l d
        b = rearrange(b, 'b n l (m d) -> m b n l d', m=self.num_b_or_c)
        c = rearrange(c, 'b n l (m d) -> m b n l d', m=self.num_b_or_c)

        if self.unified_view == False:
            # Get DAG mixers
            idx_dag_name = {0: 'top_left', 1: 'top_right', 2: 'bottom_left', 3: 'bottom_right'}
            bc_idx = 0
            dt_idx = 0 # serves as idx for both dt and dt_self
            output = torch.zeros_like(hidden_states_rearranged, device=device)
            I_matrix = torch.eye(self.height*self.image_width, device=device)

            for key in idx_dag_name.keys():
                if self.include_headnodes[key] == '1':
                    A_matrix, num_incident_edges = get_normalized_dag_mixer(expdt[dt_idx], idx_dag_name[key], device, True)
                    matrix_mixer = DAGInverse.apply(A_matrix.to(torch.float32), self.height+self.image_width) # use_fast_inverse
                    # matrix_mixer = torch.inverse(I_matrix - A_matrix)

                    # if self.normalization_mode == "dt_original":
                    summed_dt = torch.sum(dt[dt_idx], dim=0, keepdim=False) # (B, N, i, j)
                    norm_dt = summed_dt/torch.sqrt(num_incident_edges) #
                    # elif self.normalization_mode == "sqrt":
                    #     summed_dt = torch.sum(dt[dt_idx], dim=0, keepdim=False)
                    #     root_dt = norm_sqrt_mul_factor*(1-torch.exp(2*summed_dt)) # norm_sqrt_mul_factor=1.0, # < 1
                    #     norm_dt = root_dt/torch.sqrt(num_incident_edges)
                    # elif self.normalization_mode == "dt_self":
                    #     norm_dt = dt_self[dt_idx]/torch.sqrt(num_incident_edges)

                    rearrange_norm_dt = rearrange(norm_dt, 'b n i j -> b n (i j)')
                    b_bar = torch.einsum('b n l d, b n l -> b n l d', b[bc_idx], rearrange_norm_dt)
                    output = output + torch.einsum('b n l d, b n l t, b n t d, b n t h -> b n l h',
                                                    c[bc_idx], matrix_mixer, b_bar, hidden_states_rearranged)
                    bc_idx += (int(key)%2 == 1) # graphs_mode line
                    # bc_idx = bc_idx if int(key)%2 == 1 else (bc_idx + 1)%2 # graphs_mode diagonal
                    dt_idx += 1
            output = output/np.sqrt(num_matrix_mixers)
        else:
            idx_dag_name = {0: 'top_left', 1: 'top_right', 2: 'bottom_left', 3: 'bottom_right'}
            dt_idx = 0
            I_matrix = torch.eye(self.height*self.image_width, device=device)
            A_matrix = torch.zeros_like(I_matrix, device=device)

            for key in idx_dag_name.keys():
                if self.include_headnodes[key] == '1':
                    A_matrix_temp, num_incident_edges = get_normalized_dag_mixer(expdt[dt_idx], idx_dag_name[key], device, True)
                    A_matrix = A_matrix + A_matrix_temp
                    dt_idx += 1

            IA_matrix = I_matrix - A_matrix
            # if self.normalization_mode == "dt_self":
            #     leq = rearrange(dt_self[0], 'b n i j -> b n (i j)')
            #     row_sum = torch.sum(IA_matrix, dim=-1, keepdim=False) + leq
            #     IA_matrix = .95*IA_matrix/row_sum.unsqueeze(-1) # inverse_factor=0.95

            matrix_mixer = torch.inverse(IA_matrix)
            output = torch.einsum('bnld,bnlt,bntd,bnth->bnlh', c[0], matrix_mixer, b[0], hidden_states_rearranged)

        output = output + hidden_states_rearranged*self.D[None,:,None,None] # std_dev*output+hid, std_dev=1
        final_output = rearrange(output, 'b n l h -> b l (n h)')
        return final_output



In [ ]:
# @title goombalab/imagenet_experiments/chimera.py save
# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/src/models/sequence/modules/chimera.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
from functools import partial
import numpy as np
import sys
from torch_geometric.utils import to_dense_adj
from torch_geometric.data import Data

class DAGInverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, A, n):
        j = 2 ** math.ceil(math.log2(n))
        #|A|: (b n l l)
        last_A_j = A
        I_matrix =  torch.eye(A.size(-1), dtype=A.dtype, device=A.device).unsqueeze(0).unsqueeze(0)
        I_minus_A_inv = I_matrix + A
        # Compute (I+A)(I+A^2)(I+A^4)...(I+A^j)
        # TODO: Do this in log(j) steps, but for j = 32, O(j) is fine
        for _ in range(2, int(math.log2(j)) + 1):
            last_A_j = torch.matmul(last_A_j, last_A_j)
            I_minus_A_inv = torch.matmul(I_minus_A_inv, I_matrix + last_A_j)
        ctx.save_for_backward(I_minus_A_inv)
        return I_minus_A_inv

    @staticmethod
    def backward(ctx, grad_output):
        I_minus_A_inv = ctx.saved_tensors[0] # Retrieve the saved tensors from the forward pass
        grad_A = torch.matmul(I_minus_A_inv.transpose(-2, -1), torch.matmul(grad_output, I_minus_A_inv.transpose(-2, -1))) # Compute the gradient with respect to A
        return grad_A, None

class Chimera(nn.Module):
    def __init__(
        self,
        d_model,
        qk_dim=64,
        expand_factor=2,
        headdim=64,
        # graph mamba flags
        unified_view=False,
        include_headnodes="1111", # order: top_left, top_right, bottom_left, bottom_right
        debug_use_get_A_dpr=False,
        debug_store_mm=False,
        share_BC=False,
        share_BC_for_two_graphs=True,
        share_dt_for_two_graphs=True,
        share_BC_for_two_graphs_mode="line", # "diagonal"
        use_fast_inverse=True,
        dt_min_max_factor=1.0,
        dt_self_min_max_factor=1.0,
        normalization_mode="dt_original", # "dt_self", "sqrt"
        norm_sqrt_mul_factor=1.0, # < 1
        # other configs
        dt_min=0.001,
        dt_max=0.1,
        dt_init='random',
        dt_init_scale=1.0,
        dt_init_floor=1e-4,
        device=None,
        dtype=None,
        height=14, #Need to support dynamic image height
        image_width=14, #Need to support dynamic image width
    ):
        super().__init__()
        self.d_model = d_model
        self.qk_dim = qk_dim
        self.expand_factor = expand_factor
        self.headdim = headdim
        # self.max_seq_len = max_seq_len

        # graph mamba flags
        self.unified_view = unified_view
        self.include_headnodes = include_headnodes
        self.debug_use_get_A_dpr = debug_use_get_A_dpr
        self.debug_store_mm = debug_store_mm
        self.share_BC = share_BC
        self.share_BC_for_two_graphs = share_BC_for_two_graphs
        self.share_dt_for_two_graphs = share_dt_for_two_graphs
        self.share_BC_for_two_graphs_mode = share_BC_for_two_graphs_mode
        self.use_fast_inverse = use_fast_inverse
        self.dt_min = dt_min*dt_min_max_factor
        self.dt_max = dt_max*dt_min_max_factor
        self.dt_self_min = dt_min*dt_self_min_max_factor
        self.dt_self_max = dt_max*dt_self_min_max_factor
        self.normalization_mode = normalization_mode
        self.norm_sqrt_mul_factor = norm_sqrt_mul_factor

        self.d_inner = round(self.expand_factor * self.d_model)
        assert self.d_inner % self.headdim == 0
        self.num_heads = self.d_inner // self.headdim
        self.std_dev = 1.0 # This will be per method parameter
        self.tol = 1e-6
        self.inverse_factor=0.95
        self.height = height
        self.image_width = image_width

        # Initialize log dt bias
        dt_shape = (self.num_heads,)
        dt = torch.exp(torch.rand(dt_shape,) * (math.log(self.dt_max) - math.log(self.dt_min)) + math.log(self.dt_min))
        dt = torch.clamp(dt, min=dt_init_floor)
        # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        self.dt_bias = nn.Parameter(inv_dt)
        self.dt_bias._no_reinit = True
        self.dt_bias._no_weight_decay = True

        if self.normalization_mode == "dt_self":
            dt_self = torch.exp(torch.rand(dt_shape,) * (math.log(self.dt_self_max) - math.log(self.dt_self_min)) + math.log(self.dt_self_min))
            dt_self = torch.clamp(dt_self, min=dt_init_floor)
            inv_dt_self = dt_self + torch.log(-torch.expm1(-dt_self))
            self.dt_self_bias = nn.Parameter(inv_dt_self)
            self.dt_self_bias._no_reinit = True
            self.dt_self_bias._no_weight_decay = True

        # D "skip" parameter
        self.D = nn.Parameter(torch.ones(self.num_heads, device=device))
        self.D._no_reinit = True
        self.D._no_weight_decay = True

        self.unused_dt_mask = self.get_unused_dt_mask().unsqueeze(2).unsqueeze(2)

        if self.debug_store_mm:
            self.epoch=0

    def get_unused_dt_mask(self):
        # Shape of dt_rearranged: [4,2,b,h,i,j]
        # Must be used in the order [0]top_left, [1]top_right, [2]bottom_left, [3]bottom_right,
        # where each is followed by [0]left-right, [1]top-bottom

        # Visualize a Rectangle with 4 corners
        # 0 --(a)--- 1
        # |          |
        #(d)        (b)
        # |          |
        # 2 --(c)--- 3
        dt_mask = torch.ones((4, 2, self.height, self.image_width), requires_grad=False)
        # Corners and Edges:
        # For (a) remove [0][1], and [1][1]
        dt_mask[0][1][0, :] = 0
        dt_mask[1][1][0, :] = 0
        # For (b) remove [1][0], and [3][0]
        dt_mask[1][0][:, -1] = 0
        dt_mask[3][0][:, -1] = 0
        # For (c) remove [2][1], and [3][1]
        dt_mask[2][1][-1, :] = 0
        dt_mask[3][1][-1, :] = 0
        # For (d) remove [0][0], and [2][0]
        dt_mask[0][0][:, 0] = 0
        dt_mask[2][0][:, 0] = 0
        idxs = [i for i in range(4) if self.include_headnodes[i] == '1']
        return dt_mask[idxs]

    # Returns DAG mixers
    def get_normalized_dag_mixer_depr(self,  expdt,  head_node_type,  device, is_normalize=True):
        # Shape of dt_rearranged: [2, batch, num_heads, i, j]
        # where (i -> rows,j -> column)
        # 2: corresponds to left_right edges and top_bottom edges
        # |head_node_type|: values top_left, top_right, bottom_left, bottom_right
        # Exponentiate all dts
        height, width = expdt.shape[-2:]
        # num incident edges on each node
        num_incident_edges = 2*torch.ones((height, width), device=device, requires_grad=False)
        # left_right matrix ----------
        # left_right node indices
        lr_node_indices = torch.arange(height*width).reshape(height, width).to(device)

        lr_left_nodes = lr_node_indices[:, :-1].flatten()
        lr_right_nodes = lr_node_indices[:, 1:].flatten()
        lr_dt = expdt[0] # (b n i j)

        if head_node_type == 'top_left' or head_node_type == 'bottom_left':
            lr_edge_index = torch.stack([lr_left_nodes, lr_right_nodes], dim=0)
            lr_edge_attr = lr_dt[:,:,:,1:] # ensures target data dependency
            lr_edge_attr = rearrange(lr_edge_attr, "b n i j -> (i j) b n")
            num_incident_edges[:,0] = num_incident_edges[:,0] - 1
        elif head_node_type == 'top_right' or head_node_type == 'bottom_right':
            lr_edge_index = torch.stack([lr_right_nodes, lr_left_nodes], dim=0)
            lr_edge_attr = lr_dt[:,:,:,:-1] # ensures target data dependency
            lr_edge_attr = rearrange(lr_edge_attr, "b n i j -> (i j) b n")
            num_incident_edges[:,-1] = num_incident_edges[:,-1] - 1
        else: raise ValueError(f"Invalid head node: {head_node_type}")

        # top_bottom matrix ----------------
        # top_bottom node indices
        tb_node_indices = torch.arange(height*width).reshape(height, width).to(device)

        tb_top_nodes = tb_node_indices[:-1, :].flatten()
        tb_bottom_nodes = tb_node_indices[1:, :].flatten()
        tb_dt = expdt[1] # (b n i j)

        if head_node_type == 'top_left' or head_node_type == 'top_right':
            tb_edge_index = torch.stack([tb_top_nodes, tb_bottom_nodes], dim=0)
            tb_edge_attr = tb_dt[:,:,1:,:]
            tb_edge_attr = rearrange(tb_edge_attr, "b n i j -> (i j) b n")
            num_incident_edges[0,:] = num_incident_edges[0,:] - 1
        elif head_node_type == 'bottom_left' or head_node_type == 'bottom_right':
            tb_edge_index = torch.stack([tb_bottom_nodes, tb_top_nodes], dim=0)
            tb_edge_attr = tb_dt[:,:,:-1,:]
            tb_edge_attr = rearrange(tb_edge_attr, "b n i j -> (i j) b n")
            num_incident_edges[-1,:] = num_incident_edges[-1,:] - 1
        else: raise ValueError(f"Invalid head node: {head_node_type}")

        graph_data = Data(edge_index=torch.concatenate((lr_edge_index, tb_edge_index), dim=1),
            edge_attr=torch.concatenate((lr_edge_attr, tb_edge_attr), dim=0))

        adjacency_matrix = to_dense_adj(graph_data.edge_index, edge_attr=graph_data.edge_attr)
        adjacency_matrix = rearrange(adjacency_matrix[0], "l t b n -> b n l t")

        # Transpose the last two dimensions, and make it an incoming edges matrix
        adjacency_matrix = adjacency_matrix.transpose(-1, -2)
         # Divide each row by the number of non-zero values
        num_incident_edges[num_incident_edges < self.tol] = 1
        if is_normalize: final_adjacency_matrix = adjacency_matrix / torch.sqrt(rearrange(num_incident_edges, "i j -> (i j)"))[None,None,:,None]
        else: final_adjacency_matrix = adjacency_matrix
        return final_adjacency_matrix, num_incident_edges

    # Returns DAG mixers
    def get_normalized_dag_mixer(self, expdt, head_node_type, device, is_normalize=True):
        # Shape of dt_rearranged: [2, batch, num_heads, i, j]
        # where (i -> rows,j -> column)
        # 2: corresponds to left_right edges and top_bottom edges
        # |head_node_type|: values top_left, top_right, bottom_left, bottom_right

        batch, num_heads, height, width = expdt[0].shape
        seq_len = height * width

        # Matrix of all zeros (b n i j l)
        adjacency_matrix = torch.zeros((batch, num_heads, height*width, height*width), device=device)
        # num incident edges on each node
        num_incident_edges = 2*torch.ones((height, width), device=device, requires_grad=False)

        # left_right matrix ----------
        index_set_2d = torch.arange(0, seq_len).reshape(height, width).to(device)

        # left_right node dt values
        lr_dt = expdt[0] # (b n i j)

        if head_node_type == 'top_left' or head_node_type == 'bottom_left':
            left_set = index_set_2d[:, :-1].flatten()
            right_set = index_set_2d[:, 1:].flatten()
            value_set = rearrange(lr_dt, "b n i j -> b n (i j)")
            adjacency_matrix[:, :, left_set, right_set] = value_set[:, :, right_set]
            num_incident_edges[:,0] = num_incident_edges[:,0] - 1
        elif head_node_type == 'top_right' or head_node_type == 'bottom_right':
            left_set = index_set_2d[:, 1:].flatten()
            right_set = index_set_2d[:, :-1].flatten()
            value_set = rearrange(lr_dt, "b n i j -> b n (i j)")
            adjacency_matrix[:, :, left_set, right_set] = value_set[:, :, right_set]
            num_incident_edges[:,-1] = num_incident_edges[:,-1] - 1
        else: raise ValueError(f"Invalid head node: {head_node_type}")

        # # top_bottom matrix ----------------
        # top_bottom node dt values
        tb_dt = expdt[1] # (b n i j)

        if head_node_type == 'top_left' or head_node_type == 'top_right':
            left_set = index_set_2d[:-1, :].flatten()
            right_set = index_set_2d[1:, :].flatten()
            value_set = rearrange(tb_dt, "b n i j -> b n (i j)")
            adjacency_matrix[:, :, left_set, right_set] = value_set[:, :, right_set]
            num_incident_edges[0,:] = num_incident_edges[0,:] - 1
        elif head_node_type == 'bottom_left' or head_node_type == 'bottom_right':
            left_set = index_set_2d[1:, :].flatten()
            right_set = index_set_2d[:-1, :].flatten()
            value_set = rearrange(tb_dt, "b n i j -> b n (i j)")
            adjacency_matrix[:, :, left_set, right_set] = value_set[:, :, right_set]
            num_incident_edges[-1,:] = num_incident_edges[-1,:] - 1
        else: raise ValueError(f"Invalid head node: {head_node_type}")

        adjacency_matrix = adjacency_matrix.transpose(-1,-2) # Transpose the last two dimensions, and make it an incoming edges matrix
        num_incident_edges[num_incident_edges < self.tol] = 1 # Divide each row by the number of non-zero values

        if is_normalize: final_adjacency_matrix = adjacency_matrix / torch.sqrt(rearrange(num_incident_edges, "i j -> (i j)"))[None,None,:,None]
        else: final_adjacency_matrix = adjacency_matrix
        return final_adjacency_matrix, num_incident_edges


    def forward(self, hidden_states, wgt_params_data, bc, dt_self=None):
        assert self.height*self.image_width == hidden_states.shape[1]
        batch_size = hidden_states.shape[0]
        num_matrix_mixers = sum(map(int, self.include_headnodes))
        hidden_states_rearranged = rearrange(hidden_states, 'b l (n h) -> b n l h', n=self.num_heads) # [b,h,l,qk_dim]
        device = hidden_states.device
        self.unused_dt_mask = self.unused_dt_mask.to(device) #(num_matrix_mixers,2,i,j)

        if self.debug_use_get_A_dpr: get_normalized_dag_mixer = self.get_normalized_dag_mixer_depr
        else: get_normalized_dag_mixer = self.get_normalized_dag_mixer
        if self.debug_store_mm and str(device) == 'cuda:0':
            self.epoch = self.epoch + 1

        # Rearrange dt
        # (i -> rows,j -> column)

        # Rearrange b and c
        self.num_dt = sum(map(int, self.include_headnodes))
        if self.share_dt_for_two_graphs:
            assert self.share_BC_for_two_graphs == True
            self.num_dt = self.num_dt - 2
        dt = rearrange(wgt_params_data,  "b (i j) (n p q) -> p q b n i j",  i=self.height, j=self.image_width, n=self.num_heads, p=self.num_dt, q=2)

        if self.share_dt_for_two_graphs:
            if self.share_BC_for_two_graphs_mode == "line": dt = torch.repeat_interleave(dt, 2, dim=0)
            elif self.share_BC_for_two_graphs_mode == "diagonal": dt = torch.cat([dt, dt.flip(0)], dim=0)
            else: raise ValueError(f"Invalid share_BC_for_two_graphs_mode: {self.share_BC_for_two_graphs_mode}")

        dt = dt + self.dt_bias.unsqueeze(-1).unsqueeze(-1)
        dt = self.unused_dt_mask*F.softplus(dt)
        expdt = self.unused_dt_mask*torch.exp(-1*dt)

        if self.normalization_mode == "dt_self":
            dt_self = rearrange(dt_self, "b (i j) (n p) -> p b n i j", i=self.height, j=self.image_width,
                n=self.num_heads, p=1 if self.unified_view else sum(map(int, self.include_headnodes)))
            dt_self = F.softplus(dt_self + self.dt_self_bias[...,None,None])

        # Rearrange b and c
        self.num_b_or_c = 1 if self.share_BC or self.unified_view else sum(map(int, self.include_headnodes))
        if self.share_BC_for_two_graphs:
           assert self.share_BC == False
           assert  sum(map(int, self.include_headnodes)) == 4
           self.num_b_or_c = self.num_b_or_c - 2

        b,c = bc
        b = b.unsqueeze(1).repeat(1, self.num_heads, 1, 1) # shape: b n l d
        c = c.unsqueeze(1).repeat(1, self.num_heads, 1, 1) # shape: b n l d
        b = rearrange(b, 'b n l (m d) -> m b n l d', m=self.num_b_or_c)
        c = rearrange(c, 'b n l (m d) -> m b n l d', m=self.num_b_or_c)

        if self.unified_view == False:
            # Get DAG mixers
            idx_dag_name = {0: 'top_left', 1: 'top_right', 2: 'bottom_left', 3: 'bottom_right'}
            bc_idx = 0
            dt_idx = 0 # serves as idx for both dt and dt_self
            output = torch.zeros_like(hidden_states_rearranged, device=device)
            I_matrix = torch.eye(self.height*self.image_width, device=device)

            for key in idx_dag_name.keys():
                if self.include_headnodes[key] == '1':
                    A_matrix, num_incident_edges = get_normalized_dag_mixer(expdt[dt_idx], idx_dag_name[key], device, True)
                    if self.use_fast_inverse: matrix_mixer = DAGInverse.apply(A_matrix.to(torch.float32), self.height+self.image_width)
                    else: matrix_mixer = torch.inverse(I_matrix - A_matrix)

                    if self.normalization_mode == "dt_original":
                        summed_dt = torch.sum(dt[dt_idx], dim=0, keepdim=False) # (B, N, i, j)
                        norm_dt = summed_dt/torch.sqrt(num_incident_edges) #

                    elif self.normalization_mode == "sqrt":
                        summed_dt = torch.sum(dt[dt_idx], dim=0, keepdim=False)
                        root_dt = self.norm_sqrt_mul_factor*(1-torch.exp(2*summed_dt))
                        norm_dt = root_dt/torch.sqrt(num_incident_edges)
                    elif self.normalization_mode == "dt_self":
                        norm_dt = dt_self[dt_idx]/torch.sqrt(num_incident_edges)
                    else: raise ValueError(f"Invalid normalization mode: {self.normalization_mode}")

                    rearrange_norm_dt = rearrange(norm_dt, 'b n i j -> b n (i j)')
                    b_bar = torch.einsum('b n l d, b n l -> b n l d', b[bc_idx], rearrange_norm_dt)
                    output = output + torch.einsum('b n l d, b n l t, b n t d, b n t h -> b n l h',
                                                    c[bc_idx], matrix_mixer, b_bar, hidden_states_rearranged)
                    if not self.share_BC:
                        if self.share_BC_for_two_graphs:
                            if self.share_BC_for_two_graphs_mode == "line":
                                bc_idx += (int(key)%2 == 1)
                            elif self.share_BC_for_two_graphs_mode == "diagonal":
                                bc_idx = bc_idx if int(key)%2 == 1 else (bc_idx + 1)%2
                            else: raise ValueError(f"Invalid share_BC_for_two_graphs_mode: {self.share_BC_for_two_graphs_mode}")
                        else: bc_idx += 1
                    dt_idx += 1
            output = output/np.sqrt(num_matrix_mixers)
        else:
            assert self.share_BC == False
            idx_dag_name = {0: 'top_left', 1: 'top_right', 2: 'bottom_left', 3: 'bottom_right'}
            dt_idx = 0
            I_matrix = torch.eye(self.height*self.image_width, device=device)
            A_matrix = torch.zeros_like(I_matrix, device=device)

            for key in idx_dag_name.keys():
                if self.include_headnodes[key] == '1':
                    A_matrix_temp, num_incident_edges = get_normalized_dag_mixer(expdt[dt_idx], idx_dag_name[key], device, True)
                    A_matrix = A_matrix + A_matrix_temp
                    dt_idx += 1

            IA_matrix = I_matrix - A_matrix
            if self.normalization_mode == "dt_self":
                leq = rearrange(dt_self[0], 'b n i j -> b n (i j)')
                row_sum = torch.sum(IA_matrix, dim=-1, keepdim=False) + leq
                IA_matrix = self.inverse_factor*IA_matrix/row_sum.unsqueeze(-1)
            else: raise ValueError(f"Invalid normalization mode: {self.normalization_mode}")

            matrix_mixer = torch.inverse(IA_matrix)
            output = torch.einsum('b n l d, b n l t, b n t d, b n t h -> b n l h', c[0], matrix_mixer, b[0], hidden_states_rearranged)

        if self.debug_store_mm and str(device) == 'cuda:0' and self.epoch % 5000 == 1:
            # this is not mulitplied by dts: you have to do it manually
            # the constituent expdts are normalized
            np.save(f'matrix_mixer_{self.epoch}.npy', matrix_mixer.detach().cpu().numpy())
            # this is normalized using sqrt incident edges
            np.save(f'dts_{self.epoch}.npy', dt.detach().cpu().numpy())
            np.save(f'num_incident_{self.epoch}.npy', num_incident_edges.detach().cpu().numpy())

            # save B's and C's X's
            np.save(f'b_{self.epoch}.npy', b.to(torch.float32).detach().cpu().numpy())
            np.save(f'c_{self.epoch}.npy', c.to(torch.float32).detach().cpu().numpy())
            np.save(f'x_{self.epoch}.npy', hidden_states_rearranged.to(torch.float32).detach().cpu().numpy())

        output = self.std_dev*(output) + hidden_states_rearranged*self.D.unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
        final_output = rearrange(output, 'b n l h -> b l (n h)')
        return final_output



In [ ]:
# @title goombalab/language_experiments/chimera.py save
# https://github.com/goombalab/chimera/blob/main/language_experiments/chimera/modules/chimera.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from einops import rearrange, repeat

try:
    from mamba_ssm.ops.triton.layernorm_gated import RMSNorm as RMSNormGated
except ImportError:
    RMSNormGated = None

from mamba_ssm.ops.triton.ssd_combined import mamba_chunk_scan_combined


class Chimera(nn.Module):

    def __init__(
        self,
        d_model,
        d_state=64,
        d_conv=7,
        conv_init=None,
        expand=2,
        headdim=64,
        ngroups=1,
        dt_min=0.001,
        dt_max=0.1,
        dt_init_floor=1e-4,
        dt_limit=(0.0, float("inf")),
        learnable_init_states=False,
        activation="swish",
        bias=False,
        conv_bias=True,
        # Fused kernel and sharding options
        chunk_size=256,
        layer_idx=None,  # Absorb kwarg for general module
        device=None,
        dtype=None,
    ):
        factory_kwargs = {"device": device, "dtype": dtype}
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.conv_init = conv_init
        self.expand = expand
        self.d_inner = self.expand * self.d_model
        self.headdim = headdim
        self.ngroups = ngroups
        assert self.d_inner % self.headdim == 0
        self.nheads = self.d_inner // self.headdim
        self.dt_limit = dt_limit
        self.learnable_init_states = learnable_init_states
        self.activation = activation
        self.chunk_size = chunk_size
        self.layer_idx = layer_idx

        # Order: [z, x, B, C, dt]
        d_in_proj = 2 * self.d_inner + 2 * (2 * self.ngroups * self.d_state) + 2 * self.nheads
        self.in_proj = nn.Linear(self.d_model, d_in_proj, bias=bias, **factory_kwargs)

        conv_dim = self.d_inner + 2 * (2 * self.ngroups * self.d_state)
        self.conv1d = nn.Conv1d(
            in_channels=conv_dim,
            out_channels=conv_dim,
            bias=conv_bias,
            kernel_size=d_conv,
            groups=conv_dim,
            padding=d_conv // 2,
            **factory_kwargs,
        )
        if self.conv_init is not None:
            nn.init.uniform_(self.conv1d.weight, -self.conv_init, self.conv_init)
        # self.conv1d.weight._no_weight_decay = True

        if self.learnable_init_states:
            self.init_states = nn.Parameter(torch.zeros(self.nheads, self.headdim, self.d_state, **factory_kwargs))
            self.init_states._no_weight_decay = True

        self.act = nn.SiLU()

        # Initialize log dt bias
        dt = torch.exp(
            torch.rand(self.nheads, **factory_kwargs) * (math.log(dt_max) - math.log(dt_min))
            + math.log(dt_min)
        )
        dt = torch.clamp(dt, min=dt_init_floor)
        # Inverse of softplus: https://github.com/pytorch/pytorch/issues/72759
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        self.dt_bias = nn.Parameter(inv_dt)
        # Just to be explicit. Without this we already don't put wd on dt_bias because of the check
        # name.endswith("bias") in param_grouping.py
        self.dt_bias._no_weight_decay = True

        # A parameter
        A = torch.ones(self.nheads, dtype=torch.float32, device=device)
        A_log = torch.log(A).to(dtype=dtype)
        self.A_log = nn.Parameter(A_log)
        # self.register_buffer("A_log", torch.zeros(self.nheads, dtype=torch.float32, device=device), persistent=True)
        self.A_log._no_weight_decay = True

        # D "skip" parameter
        self.D = nn.Parameter(torch.ones(self.nheads, device=device))
        self.D._no_weight_decay = True

        # Extra normalization layer right before output projection
        assert RMSNormGated is not None
        self.norm = RMSNormGated(self.d_inner, eps=1e-5, norm_before_gate=True, **factory_kwargs)

        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=bias, **factory_kwargs)

    def forward(self, u, seq_idx=None): # btd
        batch, seqlen, dim = u.shape
        zxbcdt = self.in_proj(u)  # (B, L, d_in_proj)
        A = -torch.exp(self.A_log.float())  # (nheads) or (d_inner, d_state)
        initial_states = repeat(self.init_states, "... -> b ...", b=2*batch) if self.learnable_init_states else None
        dt_limit_kwargs = {} if self.dt_limit == (0.0, float("inf")) else dict(dt_limit=self.dt_limit)

        z, xBC, dt = torch.split(zxbcdt, [self.d_inner, self.d_inner + 2 * (2 * self.ngroups * self.d_state), 2 * self.nheads], dim=-1)

        dt = torch.cat((dt[:, :, :self.nheads], torch.flip(dt[:, :, self.nheads:], (1,))), dim=0)
        dt = F.softplus(dt + self.dt_bias)  # (2 * B, L, nheads)
        assert self.activation in ["silu", "swish"]

        # 1D Convolution
        xBC = self.act(self.conv1d(xBC.transpose(1,2)).transpose(1,2))  # (B, L, self.d_inner + 2 * (2 * ngroups * d_state))

        # Split into 3 main branches: X, B, C
        # These correspond to V, K, Q respectively in the SSM/attention duality
        x, BC = torch.split(xBC, [self.d_inner, 2 * (2 * self.ngroups * self.d_state)], dim=-1)
        x_og = x
        x = torch.cat((x, torch.flip(x, (1,))), dim=0)
        BC = torch.cat([BC[:, :, :2*self.ngroups*self.d_state], torch.flip(BC[:, :, 2 * self.ngroups * self.d_state:], (1,))], dim=0)
        B, C = torch.split(BC, [self.ngroups * self.d_state, self.ngroups * self.d_state], dim=-1)
        y = mamba_chunk_scan_combined(rearrange(x, "b l (h p) -> b l h p", p=self.headdim), dt, A,
            rearrange(B, "b l (g n) -> b l g n", g=self.ngroups), rearrange(C, "b l (g n) -> b l g n", g=self.ngroups),
            chunk_size=self.chunk_size, D=None, z=None, seq_idx=seq_idx, initial_states=initial_states)

        y = rearrange(y, 'b d l -> b l d').contiguous()
        y_fw, y_bw = y[:batch], torch.flip(y[batch:], (1,))
        y = y_fw + y_bw + x_og * self.D

        # Multiply "gate" branch and apply extra normalization layer
        y = self.norm(y, z)
        out = self.out_proj(y)
        return out # btd


In [ ]:
# @title configs
# https://github.com/goombalab/chimera/blob/main/language_experiments/chimera/modules/chimera.py
# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/src/models/sequence/kernels/ssm.py

# https://github.com/goombalab/chimera/blob/main/language_experiments/chimera/bert/yamls/pretrain/chimera.yaml
# mlm
# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/configs/experiment/s4nd/vit/chimera_b_16_imagenet.yaml
#   lr: 0.001
#   weight_decay: 0.05

# https://github.com/goombalab/chimera/blob/main/imagenet_experiments/configs/model/vit/chimera_b_16.yaml
patch_size: 16
d_model: 768
depth: 23
qk_dim: 64 # vs 64 for ViT-B
  headdim: 128 # 768*2/128 = 12 heads vs 12 heads for ViT-B
  unified_view: False # keep it False for now
  include_headnodes: "1111"

  debug_use_get_A_dpr: False
  debug_store_mm: False

  share_BC: False
  share_BC_for_two_graphs: True
  share_dt_for_two_graphs: False
  share_BC_for_two_graphs_mode: "diagonal" # "line" or "diagonal"
  add_fc_layers: False
  expand_factor: "2.0"
  use_fast_inverse: True
  dt_min_max_factor: 7.0 # [5, 7, 10] - For a shorter sequence length
  dt_self_min_max_factor: 1.0 # [1, 5] UNUSED
  normalization_mode: "dt_original"
  norm_sqrt_mul_factor: 1.0
